# package installation and settings

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

import os
import copy
import math

from datetime import timedelta
from scipy.stats import linregress

In [2]:
import matplotlib

print(np.__version__)
print(pd.__version__)
print(matplotlib.__version__)
print(sns.__version__)

2.2.6
2.3.3
3.10.8
0.13.2


In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# data import

In [4]:
def icu_data_processor(path, df_general_list):

  dfs = []

  for dirpath, dirnames, filenames in os.walk(path):
    for filename in filenames:
      if filename.endswith('.txt'):
        df_temp = pd.read_csv(os.path.join(dirpath, filename))
        df_temp['RecordID'] = filename.split('.')[0]
        df_index = []
        df_index.append(df_temp[df_temp['Parameter'] == 'RecordID'].index.values[0].item())
        df_index.append(df_temp[df_temp['Parameter'] == 'Age'].index.values[0].item())
        df_index.append(df_temp[df_temp['Parameter'] == 'Gender'].index.values[0].item())
        df_index.append(df_temp[df_temp['Parameter'] == 'Height'].index.values[0].item())
        df_index.append(df_temp[df_temp['Parameter'] == 'ICUType'].index.values[0].item())
        df_general_list.append(df_temp.iloc[df_index,])
        dfs.append(df_temp.drop(df_index, axis=0))

  return dfs

In [5]:
path = '/home/leewen/code/physionet'
df_general_list = []
dfs_a = icu_data_processor(path+'/data/set-a', df_general_list)
dfs_b = icu_data_processor(path+'/data/set-b', df_general_list)
dfs_c = icu_data_processor(path+'/data/set-c', df_general_list)

In [6]:
df_outcome_a = pd.read_csv(path+'/data/Outcomes-a.txt')
df_outcome_b = pd.read_csv(path+'/data/Outcomes-b.txt')
df_outcome_c = pd.read_csv(path+'/data/Outcomes-c.txt')

In [7]:
df_outcome = pd.concat([df_outcome_a, df_outcome_b, df_outcome_c], axis=0).reset_index(drop=True)

In [8]:
df_outcome.head(10)

,RecordID,SAPS-I,SOFA,Length_of_stay,Survival,In-hospital_death
0,132539,6,1,5,-1,0
1,132540,16,8,8,-1,0
2,132541,21,11,19,-1,0
3,132543,7,1,9,575,0
4,132545,17,2,4,918,0
5,132547,14,11,6,1637,0
6,132548,14,4,9,-1,0
7,132551,19,8,6,5,1
8,132554,11,0,17,38,0
9,132555,14,6,8,-1,0


In [9]:
df_outcome['In-hospital_death'].value_counts()

In-hospital_death
0    10293
1     1707
Name: count, dtype: int64

In [10]:
print(len(dfs_a))
print(len(dfs_b))
print(len(dfs_c))
print(len(df_general_list))
print(df_outcome_a.shape)
print(df_outcome_b.shape)
print(df_outcome_c.shape)
print(df_outcome.shape)

4000
4000
4000
12000
(4000, 6)
(4000, 6)
(4000, 6)
(12000, 6)


# patient demographic data

In [11]:
def process_general_data(df_general_list):

  df_list = []

  for i in range(len(df_general_list)):
    temp = df_general_list[i].T.loc['Value',].values.tolist()
    temp[0] = int(temp[0])
    df_list.append(temp)
  df_general = pd.DataFrame(df_list)
  df_general.columns = ['RecordID', 'Age', 'Gender', 'Height', 'ICUType']

  return df_general

In [12]:
df_general = process_general_data(df_general_list)

In [13]:
df_general.head(10)

,RecordID,Age,Gender,Height,ICUType
0,137188,22.0,0.0,167.6,3.0
1,140351,72.0,1.0,177.8,2.0
2,141985,61.0,1.0,-1.0,1.0
3,136263,67.0,1.0,167.6,2.0
4,141669,50.0,0.0,-1.0,3.0
5,133368,83.0,0.0,152.4,2.0
6,132765,90.0,1.0,-1.0,4.0
7,138320,66.0,1.0,175.3,1.0
8,136295,61.0,1.0,-1.0,4.0
9,134326,59.0,1.0,-1.0,4.0


In [14]:
df_general.shape

(12000, 5)

In [15]:
df_patient =  pd.merge(left=df_general, right=df_outcome, on='RecordID', how='inner')

In [16]:
df_patient.head(10)

,RecordID,Age,Gender,Height,ICUType,SAPS-I,SOFA,Length_of_stay,Survival,In-hospital_death
0,137188,22.0,0.0,167.6,3.0,15,16,13,-1,0
1,140351,72.0,1.0,177.8,2.0,22,13,15,1007,0
2,141985,61.0,1.0,-1.0,1.0,8,1,12,257,0
3,136263,67.0,1.0,167.6,2.0,17,9,9,-1,0
4,141669,50.0,0.0,-1.0,3.0,7,0,4,-1,0
5,133368,83.0,0.0,152.4,2.0,16,9,17,-1,0
6,132765,90.0,1.0,-1.0,4.0,20,10,18,120,0
7,138320,66.0,1.0,175.3,1.0,8,2,3,-1,0
8,136295,61.0,1.0,-1.0,4.0,4,2,13,-1,0
9,134326,59.0,1.0,-1.0,4.0,12,7,13,-1,0


In [17]:
df_patient.shape

(12000, 10)

In [18]:
df_patient.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   RecordID           12000 non-null  int64  
 1   Age                12000 non-null  float64
 2   Gender             12000 non-null  float64
 3   Height             12000 non-null  float64
 4   ICUType            12000 non-null  float64
 5   SAPS-I             12000 non-null  int64  
 6   SOFA               12000 non-null  int64  
 7   Length_of_stay     12000 non-null  int64  
 8   Survival           12000 non-null  int64  
 9   In-hospital_death  12000 non-null  int64  
dtypes: float64(4), int64(6)
memory usage: 937.6 KB


In [19]:
df_patient.describe()

,RecordID,Age,Gender,Height,ICUType,SAPS-I,SOFA,Length_of_stay,Survival,In-hospital_death
count,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000
mean,147744.694750,64.529333,0.559583,88.159075,2.778333,14.308917,6.405917,13.420167,133.932417,0.142250
std,8787.621488,17.175677,0.498468,86.081235,1.017571,6.043346,4.220225,12.812649,372.716220,0.349321
min,132539.000000,15.000000,-1.000000,-1.000000,1.000000,-1.000000,-1.000000,-1.000000,-23.000000,0.000000
25%,140100.750000,53.000000,0.000000,-1.000000,2.000000,11.000000,3.000000,6.000000,-1.000000,0.000000
50%,147739.000000,67.000000,1.000000,152.400000,3.000000,15.000000,6.000000,10.000000,-1.000000,0.000000
75%,155309.750000,78.000000,1.000000,170.200000,4.000000,18.000000,9.000000,16.000000,19.000000,0.000000
max,163037.000000,90.000000,1.000000,462.300000,4.000000,38.000000,22.000000,295.000000,2664.000000,1.000000


# missing value and outlier - patient demographic data

In [20]:
df_patient['Gender'].value_counts(dropna=False)

Gender
 1.0    6727
 0.0    5261
-1.0      12
Name: count, dtype: int64

In [21]:
df_patient['ICUType'].value_counts(dropna=False)

ICUType
3.0    4293
4.0    3408
2.0    2530
1.0    1769
Name: count, dtype: int64

In [22]:
df_patient['SAPS-I'].value_counts(dropna=False)

SAPS-I
 16    856
 15    848
 14    809
 18    774
 17    752
 13    746
 12    731
 19    674
 11    619
 10    607
 20    592
-1     518
 9     506
 21    473
 8     395
 22    366
 7     292
 23    272
 6     243
 24    213
 5     169
 25    115
 4      99
 26     82
 27     61
 28     51
 3      48
 2      27
 29     21
 30     16
 32      7
 33      6
 1       6
 31      3
 34      1
 0       1
 38      1
Name: count, dtype: int64

In [23]:
df_patient['SOFA'].value_counts(dropna=False)

SOFA
 8     1094
 7     1082
 1     1009
 6      942
 9      914
 5      897
 2      894
 4      807
 3      765
 10     733
 11     617
 12     502
-1      403
 13     383
 0      365
 14     247
 15     142
 16      81
 17      48
 18      36
 20      13
 22      11
 19      10
 21       5
Name: count, dtype: int64

In [24]:
df_patient[df_patient['Gender'] < 0].shape

(12, 10)

In [25]:
df_patient[df_patient['SAPS-I'] < 0].shape

(518, 10)

In [26]:
df_patient[df_patient['SOFA'] < 0].shape

(403, 10)

In [27]:
df_patient.loc[df_patient['Gender'] < 0, 'Gender'] = np.nan

In [28]:
df_patient.loc[df_patient['SAPS-I'] < 0, 'SAPS-I'] = np.nan

In [29]:
df_patient.loc[df_patient['SOFA'] < 0, 'SOFA'] = np.nan

In [30]:
df_patient[df_patient['Age'] < 0].shape

(0, 10)

In [31]:
df_patient[df_patient['Age'] > 130].shape

(0, 10)

In [32]:
df_patient[df_patient['Height'] < 90].shape

(5743, 10)

In [33]:
df_patient.loc[df_patient['Height'] < 10, 'Height'] = np.nan

In [34]:
df_patient[df_patient['Height'] > 250].shape

(11, 10)

In [35]:
df_patient.loc[df_patient['Height'] > 300, 'Height'] = np.nan

In [36]:
print(df_patient[df_patient['Length_of_stay'] == -1].shape)
print(df_patient[df_patient['Length_of_stay'] == 0].shape)
print(df_patient[df_patient['Length_of_stay'] == 1].shape)

(167, 10)
(0, 10)
(5, 10)


In [37]:
df_patient.loc[df_patient['Length_of_stay'] < 2, 'Length_of_stay'] = np.nan

In [38]:
print(df_patient[df_patient['Survival'] < -1].shape)
print(df_patient[df_patient['Survival'] == -1].shape)
print(df_patient[df_patient['Survival'] == 0].shape)
print(df_patient[df_patient['Survival'] == 1].shape)

(1, 10)
(7410, 10)
(6, 10)
(85, 10)


In [39]:
df_patient.loc[df_patient['Survival'] < -1, 'Survival'] = np.nan
df_patient.loc[df_patient['Survival'] == 0, 'Survival'] = np.nan
df_patient.loc[df_patient['Survival'] == 1, 'Survival'] = np.nan

In [40]:
icu_dummies = pd.get_dummies(df_patient['ICUType'], prefix='ICUType')
df_patient = pd.concat([df_patient, icu_dummies], axis=1)
df_patient.drop('ICUType', axis=1, inplace=True)

In [41]:
df_patient.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   RecordID           12000 non-null  int64  
 1   Age                12000 non-null  float64
 2   Gender             11988 non-null  float64
 3   Height             6261 non-null   float64
 4   SAPS-I             11482 non-null  float64
 5   SOFA               11597 non-null  float64
 6   Length_of_stay     11828 non-null  float64
 7   Survival           11908 non-null  float64
 8   In-hospital_death  12000 non-null  int64  
 9   ICUType_1.0        12000 non-null  bool   
 10  ICUType_2.0        12000 non-null  bool   
 11  ICUType_3.0        12000 non-null  bool   
 12  ICUType_4.0        12000 non-null  bool   
dtypes: bool(4), float64(7), int64(2)
memory usage: 890.8 KB


In [42]:
df_patient.describe()

,RecordID,Age,Gender,Height,SAPS-I,SOFA,Length_of_stay,Survival,In-hospital_death
count,12000.000000,12000.000000,11988.000000,6261.000000,11482.000000,11597.000000,11828.000000,11908.000000,12000.000000
mean,147744.694750,64.529333,0.561144,169.146782,14.999565,6.663275,13.629016,134.961958,0.142250
std,8787.621488,17.175677,0.496268,13.298488,5.207572,4.056709,12.786962,373.968472,0.349321
min,132539.000000,15.000000,0.000000,12.700000,0.000000,0.000000,2.000000,-1.000000,0.000000
25%,140100.750000,53.000000,0.000000,162.600000,11.000000,3.000000,6.000000,-1.000000,0.000000
50%,147739.000000,67.000000,1.000000,170.200000,15.000000,7.000000,10.000000,-1.000000,0.000000
75%,155309.750000,78.000000,1.000000,177.800000,19.000000,9.000000,17.000000,20.000000,0.000000
max,163037.000000,90.000000,1.000000,210.800000,38.000000,22.000000,295.000000,2664.000000,1.000000


# icu data time format


In [43]:
dfs_a1 = []
dfs_b1 = []
dfs_c1 = []

In [44]:
def string_to_datetime(time_string):
  if ':' not in time_string:
    raise ValueError('Invalid time format.')

  try:
    hour_string, minute_string = time_string.split(':')
    hour = int(hour_string)
    minute = int(minute_string)
    duration = timedelta(hours=hour, minutes=minute)
    total_minute = int(duration.total_seconds() / 60)
    return total_minute
  except ValueError as e:
    print(f'Could not parse {time_string}.')
    print(f'Error: {e}')
    return None

In [45]:
def time_data_processor(dfs_list1, dfs_list2):
  for i in range(len(dfs_list1)):
    df_temp = dfs_list1[i].sort_values(['Parameter', 'Time']).reset_index(drop=True)
    df_temp['Time_Minute'] = df_temp['Time'].apply(string_to_datetime)
    df_temp_col = ['RecordID', 'Parameter', 'Value', 'Time', 'Time_Minute']
    df_temp = df_temp[df_temp_col]
    dfs_list2.append(df_temp)
  return dfs_list2

In [46]:
dfs_a1 = time_data_processor(dfs_a, dfs_a1)
dfs_b1 = time_data_processor(dfs_b, dfs_b1)
dfs_c1 = time_data_processor(dfs_c, dfs_c1)
dfs = dfs_a1+dfs_b1+dfs_c1
print(len(dfs_a1))
print(len(dfs_b1))
print(len(dfs_c1))
print(len(dfs))

4000
4000
4000
12000


# missing value and outlier - icu data

## missing value

In [47]:
col_icu = ['Albumin', 'ALP', 'ALT', 'AST', 'Bilirubin', 'BUN', 'Cholesterol', 'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT', 'HR', 'K', 'Lactate', 'Mg', 'MAP', 'MechVent', 'Na', 'NIDiasABP', 'NIMAP', 'NISysABP', 'PaCO2', 'PaO2', 'pH', 'Platelets', 'RespRate', 'SaO2', 'SysABP', 'Temp', 'TroponinI', 'TroponinT', 'Urine', 'WBC', 'Weight']

In [48]:
dfs_clean = []
index_clean = []

def missing_cleaner(df, i):
  if df['Parameter'].isna().sum() != 0:
    dfs_clean.append(df.dropna(subset=['Parameter']))
    index_clean.append(i)
  else:
    dfs_clean.append(df)

In [49]:
for i in range(len(dfs)):
  missing_cleaner(dfs[i], i)

In [50]:
index_clean[0:10]

[8003, 8006, 8007, 8008, 8013, 8014, 8015, 8016, 8019, 8021]

In [51]:
dfs[8000].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 675 entries, 0 to 674
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   RecordID     675 non-null    object 
 1   Parameter    675 non-null    object 
 2   Value        675 non-null    float64
 3   Time         675 non-null    object 
 4   Time_Minute  675 non-null    int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 26.5+ KB


In [52]:
dfs_clean[8000].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 675 entries, 0 to 674
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   RecordID     675 non-null    object 
 1   Parameter    675 non-null    object 
 2   Value        675 non-null    float64
 3   Time         675 non-null    object 
 4   Time_Minute  675 non-null    int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 26.5+ KB


## outlier

In [53]:
def outlier_detector(df, value_dict):

  df_result = []

  for i in range(len(value_dict)):
    col = value_dict[i]['col']
    value_lower = value_dict[i]['value_lower']
    value_upper = value_dict[i]['value_upper']
    df_temp = df[df['Parameter']==col].reset_index(drop=True)
    for j in range(df_temp.shape[0]):
      if df_temp['Value'][j] > value_upper or df_temp['Value'][j] < value_lower:
        df_result.append(df_temp.loc[j,:].T.to_dict())

  return df_result

In [54]:
value_dict_icu = [
    {'col': 'Albumin', 'value_lower': 0.5, 'value_upper': 7.0},
    {'col': 'ALP', 'value_lower': 5, 'value_upper': 5000},
    {'col': 'ALT', 'value_lower': 0, 'value_upper': 20000},
    {'col': 'AST', 'value_lower': 0, 'value_upper': 20000},
    {'col': 'Bilirubin', 'value_lower': 0.1, 'value_upper': 100.0},
    {'col': 'BUN', 'value_lower': 2, 'value_upper': 300},
    {'col': 'Cholesterol', 'value_lower': 10, 'value_upper': 800},
    {'col': 'Creatinine', 'value_lower': 0, 'value_upper': 30.0},
    {'col': 'DiasABP', 'value_lower': 15, 'value_upper': 150},
    {'col': 'FiO2', 'value_lower': 0.21, 'value_upper': 1.0},
    {'col': 'GCS', 'value_lower': 3, 'value_upper': 15},
    {'col': 'Glucose', 'value_lower': 20, 'value_upper': 1500},
    {'col': 'HCO3', 'value_lower': 5, 'value_upper': 60},
    {'col': 'HCT', 'value_lower': 10, 'value_upper': 70},
    {'col': 'HR', 'value_lower': 20, 'value_upper': 300},
    {'col': 'K', 'value_lower': 1.5, 'value_upper': 10.0},
    {'col': 'Lactate', 'value_lower': 0.1, 'value_upper': 35.0},
    {'col': 'MAP', 'value_lower': 10, 'value_upper': 170},
    {'col': 'MechVent', 'value_lower': 0, 'value_upper': 1},
    {'col': 'Mg', 'value_lower': 0.3, 'value_upper': 6.0},
    {'col': 'Na', 'value_lower': 100, 'value_upper': 170},
    {'col': 'NIDiasABP', 'value_lower': 15, 'value_upper': 150},
    {'col': 'NIMAP', 'value_lower': 20, 'value_upper': 170},
    {'col': 'NISysABP', 'value_lower': 40, 'value_upper': 250},
    {'col': 'PaCO2', 'value_lower': 10, 'value_upper': 100},
    {'col': 'PaO2', 'value_lower': 20, 'value_upper': 700},
    {'col': 'pH', 'value_lower': 6.8, 'value_upper': 7.8},
    {'col': 'Platelets', 'value_lower': 10, 'value_upper': 1500},
    {'col': 'RespRate', 'value_lower': 5, 'value_upper': 60},
    {'col': 'SaO2', 'value_lower': 50, 'value_upper': 100},
    {'col': 'SysABP', 'value_lower': 40, 'value_upper': 250},
    {'col': 'Temp', 'value_lower': 25.0, 'value_upper': 42.5},
    {'col': 'TroponinI', 'value_lower': 0, 'value_upper': 50},
    {'col': 'TroponinT', 'value_lower': 0, 'value_upper': 50},
    {'col': 'Urine', 'value_lower': 0, 'value_upper': 20000},
    {'col': 'WBC', 'value_lower': 0.5, 'value_upper': 800},
    {'col': 'Weight', 'value_lower': 20, 'value_upper': 500}]

In [55]:
value_dict_normal = [
    {'col': 'Albumin', 'value_lower': 3.5, 'value_upper': 5.0},
    {'col': 'ALP', 'value_lower': 44, 'value_upper': 147},
    {'col': 'ALT', 'value_lower': 7, 'value_upper': 56},
    {'col': 'AST', 'value_lower': 5, 'value_upper': 40},
    {'col': 'Bilirubin', 'value_lower': 0.2, 'value_upper': 1.2},
    {'col': 'BUN', 'value_lower': 6, 'value_upper': 20},
    {'col': 'Cholesterol', 'value_lower': 200, 'value_upper': 200},
    {'col': 'Creatinine', 'value_lower': 0.5, 'value_upper': 1.2},
    {'col': 'DiasABP', 'value_lower': 80, 'value_upper': 80},
    {'col': 'FiO2', 'value_lower': 0.21, 'value_upper': 0.21},
    {'col': 'GCS', 'value_lower': 3, 'value_upper': 15},
    {'col': 'Glucose', 'value_lower': 70, 'value_upper': 99},
    {'col': 'HCO3', 'value_lower': 22, 'value_upper': 29},
    {'col': 'HCT', 'value_lower': 37, 'value_upper': 54},
    {'col': 'HR', 'value_lower': 60, 'value_upper': 100},
    {'col': 'K', 'value_lower': 3.5, 'value_upper': 5.0},
    {'col': 'Lactate', 'value_lower': 0.5, 'value_upper': 1.0},
    {'col': 'MAP', 'value_lower': 70, 'value_upper': 120},
    {'col': 'MechVent', 'value_lower': 0, 'value_upper': 0},
    {'col': 'Mg', 'value_lower': 0.70, 'value_upper': 1.10},
    {'col': 'Na', 'value_lower': 135, 'value_upper': 145},
    {'col': 'NIDiasABP', 'value_lower': 80, 'value_upper': 80},
    {'col': 'NIMAP', 'value_lower': 70, 'value_upper': 120},
    {'col': 'NISysABP', 'value_lower': 120, 'value_upper': 120},
    {'col': 'PaCO2', 'value_lower': 35, 'value_upper': 45},
    {'col': 'PaO2', 'value_lower': 80, 'value_upper': 100},
    {'col': 'pH', 'value_lower': 7.35, 'value_upper': 7.45},
    {'col': 'Platelets', 'value_lower': 150, 'value_upper': 450},
    {'col': 'RespRate', 'value_lower': 12, 'value_upper': 20},
    {'col': 'SaO2', 'value_lower': 95, 'value_upper': 100},
    {'col': 'SysABP', 'value_lower': 120, 'value_upper': 120},
    {'col': 'Temp', 'value_lower': 36.5, 'value_upper': 37.5},
    {'col': 'TroponinI', 'value_lower': 0, 'value_upper': 0.01},
    {'col': 'TroponinT', 'value_lower': 0, 'value_upper': 0.01},
    {'col': 'Urine', 'value_lower': 500, 'value_upper': 2000},
    {'col': 'WBC', 'value_lower': 4.5, 'value_upper': 11.0},
    {'col': 'Weight', 'value_lower': 50, 'value_upper': 60}]

In [56]:
healthy_baselines = {
    'Albumin': 4.0,
    'ALP': 75.0,
    'ALT': 25.0,
    'AST': 25.0,
    'Bilirubin': 0.7,
    'BUN': 15.0,
    'Cholesterol': 150.0,
    'Creatinine': 1.0,
    'DiasABP': 70.0,
    'FiO2': 0.21,
    'GCS': 15.0,
    'Glucose': 90.0,
    'HCO3': 24.0,
    'HCT': 40.0,
    'HR': 75.0,
    'K': 4.0,
    'Lactate': 1.0,
    'Mg': 2.0,
    'MAP': 85.0,
    'Na': 140.0,
    'NIDiasABP': 70.0,
    'NIMAP': 85.0,
    'NISysABP': 120.0,
    'PaCO2': 40.0,
    'PaO2': 90.0,
    'pH': 7.4,
    'Platelets': 250.0,
    'RespRate': 16.0,
    'SaO2': 98.0,
    'SysABP': 120.0,
    'Temp': 37.0,
    'TroponinI': 0.01,
    'TroponinT': 0.01,
    'Urine': 50.0,
    'WBC': 7.0,
    'Weight': 80.0}

In [57]:
df_outlier_combine = []

for i in range(len(dfs_clean)):
  df_outlier_temp = outlier_detector(dfs_clean[i], value_dict_icu)
  if len(df_outlier_temp) != 0:
    df_outlier_combine.append(df_outlier_temp)


df_outlier = [item for sublist in df_outlier_combine for item in sublist]
df_outlier = pd.DataFrame(df_outlier, columns = ['RecordID', 'Parameter', 'Value', 'Time', 'Time_Minute'])
df_outlier.shape

(9020, 5)

In [58]:
def outlier_to_missing(df, value_dict):
  df_new = copy.deepcopy(df)
  for i in range(len(value_dict)):
    col = value_dict[i]['col']
    value_lower = value_dict[i]['value_lower']
    value_upper = value_dict[i]['value_upper']
    if len(df_new[df_new['Parameter'] == col]) == 0:
      pass
    else:
      cond_1 = df_new['Parameter'] == col
      cond_2 = df_new['Value'] < value_lower
      cond_3 = df_new['Value'] > value_upper
      df_new.loc[cond_1 & cond_2, 'Value'] = np.nan
      df_new.loc[cond_1 & cond_3, 'Value'] = np.nan
  return df_new

In [59]:
dfs_nan = []

for i in range(len(dfs_clean)):
  dfs_nan.append(outlier_to_missing(dfs_clean[i], value_dict_icu))

In [60]:
count = 0
index_nan = []

for i in range(len(dfs_nan)):
    count += dfs_nan[i].isnull().sum().sum()
    if dfs_nan[i].isnull().sum().sum() != 0:
      index_nan.append(i)
print(count)

9020


In [61]:
index_nan[0:10]

[7, 9, 17, 20, 27, 28, 39, 40, 41, 42]

In [62]:
dfs_nan[3]['Value']


0       11.00
1       13.00
2       20.00
3        0.60
4        0.70
5        1.10
6       58.00
7       58.00
8       52.00
9       56.00
10      67.00
11      66.00
12      55.00
13      67.00
14      48.00
15      59.00
16      59.00
17      58.00
18      61.00
19      66.00
20      69.00
21      68.00
22      63.00
23      69.00
24      64.00
25      64.00
26      65.00
27      60.00
28      57.00
29      60.00
30      61.00
31      66.00
32      60.00
33      56.00
34       1.00
35       0.50
36       0.50
37       0.50
38       3.00
39       7.00
40      15.00
41      15.00
42      15.00
43      15.00
44      15.00
45     118.00
46     125.00
47      27.00
48      28.00
49      29.00
50      28.90
51      30.20
52      31.90
53      27.40
54      60.00
55      61.00
56      72.00
57      72.00
58      72.00
59      72.00
60      89.00
61      87.00
62      85.00
63      79.00
64      79.00
65      79.00
66      79.00
67      79.00
68      76.00
69      76.00
70      79.00
71    

In [63]:
def nan_filler(df, col):
  dfs_processed = []

  for i in col:
    df_temp = copy.deepcopy(df[df['Parameter'] == i])
    if len(df_temp) == 0:
      continue
    elif len(df_temp) == 1:
      dfs_processed.append(df_temp)
    else:
      df_temp_fill = df_temp.ffill()
      dfs_processed.append(df_temp_fill)

  df_fillna = pd.concat(dfs_processed, ignore_index=True)
  return df_fillna.reset_index(drop=True)

In [64]:
dfs_nan[6].isnull().sum()

RecordID       0
Parameter      0
Value          0
Time           0
Time_Minute    0
dtype: int64

In [65]:
df = nan_filler(dfs_nan[6], col_icu)
df.isnull().sum()

RecordID       0
Parameter      0
Value          0
Time           0
Time_Minute    0
dtype: int64

In [66]:
dfs_filled = []

for i in range(len(dfs_nan)):
    dfs_filled.append(nan_filler(dfs_nan[i], col_icu))

print(len(dfs_filled))

12000


In [67]:
count = 0

for i in range(len(dfs_filled)):
    count += dfs_filled[i].isnull().sum().sum()

print(count)

4334


In [68]:
dfs_filled[8].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 436 entries, 0 to 435
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   RecordID     436 non-null    object 
 1   Parameter    436 non-null    object 
 2   Value        436 non-null    float64
 3   Time         436 non-null    object 
 4   Time_Minute  436 non-null    int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 17.2+ KB


# data frame

In [69]:
df_temp = dfs_filled[2]
df_temp

,RecordID,Parameter,Value,Time,Time_Minute
0,141985,BUN,9.00,13:39,819
1,141985,BUN,12.00,21:56,1316
2,141985,BUN,8.00,36:03,2163
3,141985,Creatinine,0.60,13:39,819
4,141985,Creatinine,0.60,21:56,1316
5,141985,Creatinine,0.40,36:03,2163
6,141985,DiasABP,56.00,29:11,1751
7,141985,DiasABP,54.00,29:26,1766
8,141985,DiasABP,54.00,30:26,1826
9,141985,DiasABP,61.00,31:26,1886


# helper function

In [70]:
df_sub = df_temp[df_temp['Parameter'] == 'DiasABP'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,141985,DiasABP,56.0,29:11,1751
1,141985,DiasABP,54.0,29:26,1766
2,141985,DiasABP,54.0,30:26,1826
3,141985,DiasABP,61.0,31:26,1886
4,141985,DiasABP,51.0,32:26,1946
5,141985,DiasABP,52.0,33:26,2006
6,141985,DiasABP,59.0,34:26,2066
7,141985,DiasABP,50.0,35:26,2126
8,141985,DiasABP,52.0,36:26,2186
9,141985,DiasABP,48.0,37:26,2246


In [71]:
def find_max_duration(df_data, threshold, condition_type='above'):

  if condition_type.lower() not in ['above', 'below']:
    raise ValueError('condition_type must be "above" or "below".')

  max_duration = 0
  max_begin = -1
  max_end = -1

  current_duration = 0
  current_begin = -1

  for i, value in enumerate(df_data['Value']):

    is_condition_met = False
    if condition_type.lower() == 'above':
      is_condition_met = (value >= threshold)
    elif condition_type.lower() == 'below':
      is_condition_met = (value <= threshold)

    if is_condition_met:
      if current_duration == 0:
        current_begin = i
      current_duration += 1
    else:
      if current_duration > max_duration:
        max_duration = current_duration
        max_begin = current_begin
        max_end = i - 1

      current_duration = 0
      current_begin = -1

  if current_duration > max_duration:
    max_duration = current_duration
    max_begin = current_begin
    max_end = len(df_data['Value']) - 1

  return max_duration, max_begin, max_end

In [72]:
threshold = 80
condition_type = 'below'
max_duration, max_begin, max_end = find_max_duration(df_sub, threshold, condition_type)
print(max_duration, max_begin, max_end)

20 0 19


In [73]:
def find_time_duration(df_data, max_begin, max_end):
  time_begin = df_data['Time_Minute'].iloc[max_begin]
  time_end = df_data['Time_Minute'].iloc[max_end]
  time_duration = time_end - time_begin
  return time_duration, time_begin, time_end

In [74]:
time_duration, time_begin, time_end = find_time_duration(df_sub, max_begin, max_end)
print(time_duration, time_begin, time_end)

1095 1751 2846


In [75]:
df_sub = df_temp[df_temp['Parameter'] == 'BUN'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,141985,BUN,9.0,13:39,819
1,141985,BUN,12.0,21:56,1316
2,141985,BUN,8.0,36:03,2163


In [76]:
df_sub2 = df_temp[df_temp['Parameter'] == 'Creatinine'].reset_index(drop=True)
df_sub2

,RecordID,Parameter,Value,Time,Time_Minute
0,141985,Creatinine,0.6,13:39,819
1,141985,Creatinine,0.6,21:56,1316
2,141985,Creatinine,0.4,36:03,2163


In [77]:
def find_max_bcr(df_sub, df_sub2):

  if len(df_sub2) == 0:
    return np.nan
  elif len(df_sub2) == 1:
    if df_sub2.isnull().sum().sum() != 0:
      return np.nan
    else:
      if df_sub2['Value'].iloc[0] == 0:
        return np.nan
      bcr = df_sub['Value'].iloc[0]/df_sub2['Value'].iloc[0]
      return bcr
  else:

    def aggregator(series):
      if (series == 0).any():
        return series.max()
      else:
        return series.mean()
    bun_series = df_sub.groupby('Time_Minute').agg({'Time_Minute': 'min', 'Value': aggregator}).set_index('Time_Minute').sort_index()
    cr_series = df_sub2.groupby('Time_Minute').agg({'Time_Minute': 'min', 'Value': aggregator}).set_index('Time_Minute').sort_index()

    combined_index = bun_series.index.union(cr_series.index)

    bun_aligned = bun_series.reindex(combined_index)
    cr_aligned = cr_series.reindex(combined_index)

    bun_imputed = bun_aligned.ffill()
    cr_imputed = cr_aligned.ffill()

    bun_imputed = bun_imputed.bfill()
    cr_imputed = cr_imputed.bfill()

    bcr_ratio_series = pd.Series(np.where(cr_imputed['Value'] == 0, np.nan, bun_imputed['Value']/cr_imputed['Value']))
    bcr = bcr_ratio_series.max()
    return bcr

In [78]:
find_max_bcr(df_sub, df_sub2)

np.float64(20.0)

In [79]:
df_temp = dfs_filled[0]

In [80]:
df_sub = df_temp[df_temp['Parameter'] == 'Lactate'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Lactate,1.5,07:35,455
1,137188,Lactate,1.5,09:33,573
2,137188,Lactate,1.8,15:08,908
3,137188,Lactate,1.4,22:18,1338
4,137188,Lactate,1.4,35:50,2150
5,137188,Lactate,1.4,46:43,2803


In [81]:
def find_lcr(df_sub, hour):
  TIME_INITIAL = df_sub['Time_Minute'].iloc[0]
  TARGET_TIME = hour*60

  initial_value = df_sub['Value'].iloc[0]


  if initial_value == 0:
    lcr = np.nan
  elif len(df_sub['Time_Minute'][df_sub['Time_Minute'] > TARGET_TIME]) == 0:
    lcr = np.nan
  else:
    target_index = df_sub['Time_Minute'][df_sub['Time_Minute'] > TARGET_TIME].idxmin()
    final_value = df_sub.loc[target_index]['Value']
    lcr = (initial_value-final_value)/initial_value*100

  return lcr

In [82]:
find_lcr(df_sub, 6)

np.float64(0.0)

In [83]:
find_lcr(df_sub, 12)

np.float64(-20.000000000000004)

In [84]:
df_sub = df_temp[df_temp['Parameter'] == 'PaO2'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,PaO2,187.0,08:33,513
1,137188,PaO2,142.0,14:08,848
2,137188,PaO2,111.0,21:18,1278
3,137188,PaO2,115.0,26:49,1609
4,137188,PaO2,73.0,34:50,2090
5,137188,PaO2,160.0,36:46,2206
6,137188,PaO2,137.0,45:43,2743


In [85]:
df_sub2 = df_temp[df_temp['Parameter'] == 'FiO2'].reset_index(drop=True)
df_sub2

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,FiO2,1.0,02:04,124
1,137188,FiO2,1.0,05:47,347
2,137188,FiO2,0.8,08:34,514
3,137188,FiO2,0.8,11:43,703
4,137188,FiO2,0.7,14:29,869
5,137188,FiO2,0.7,18:44,1124
6,137188,FiO2,0.7,20:09,1209
7,137188,FiO2,0.7,23:10,1390
8,137188,FiO2,0.7,27:29,1649
9,137188,FiO2,0.6,28:29,1709


In [86]:
def find_min_pfr(df_sub, df_sub2):

  if len(df_sub2) == 0:
    return np.nan
  elif len(df_sub2) == 1:
    if df_sub2.isnull().sum().sum() != 0:
      return np.nan
    else:
      if df_sub2['Value'].iloc[0] == 0:
        return np.nan
      pfr = df_sub['Value'].iloc[0]/df_sub2['Value'].iloc[0]
      return pfr
  else:

    def aggregator(series):
      if (series == 0).any():
        return series.max()
      else:
        return series.mean()

    pao2_series = df_sub.groupby('Time_Minute').agg({'Time_Minute': 'min', 'Value': aggregator}).set_index('Time_Minute').sort_index()
    fio2_series = df_sub2.groupby('Time_Minute').agg({'Time_Minute': 'min', 'Value': aggregator}).set_index('Time_Minute').sort_index()

    combined_index = pao2_series.index.union(fio2_series.index)

    pao2_aligned = pao2_series.reindex(combined_index)
    fio2_aligned = fio2_series.reindex(combined_index)

    pao2_imputed = pao2_aligned.ffill()
    fio2_imputed = fio2_aligned.ffill()

    pao2_imputed = pao2_imputed.bfill()
    fio2_imputed = fio2_imputed.bfill()

    pfr_ratio_series = pd.Series(np.where(fio2_imputed['Value'] == 0, np.nan, pao2_imputed['Value']/fio2_imputed['Value']))
    pfr = pfr_ratio_series.min()

    return pfr

In [87]:
find_min_pfr(df_sub, df_sub2)

np.float64(121.66666666666667)

In [88]:
df_sub = df_temp[df_temp['Parameter'] == 'Urine'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Urine,200.0,02:14,134
1,137188,Urine,380.0,04:29,269
2,137188,Urine,225.0,05:29,329
3,137188,Urine,150.0,06:29,389
4,137188,Urine,125.0,08:29,509
5,137188,Urine,100.0,10:29,629
6,137188,Urine,80.0,12:29,749
7,137188,Urine,75.0,14:29,869
8,137188,Urine,175.0,16:29,989
9,137188,Urine,125.0,17:29,1049


In [89]:
df_sub2 = df_temp[df_temp['Parameter'] == 'Weight'].reset_index(drop=True)
df_sub2

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Weight,59.6,00:00,0
1,137188,Weight,59.6,02:14,134
2,137188,Weight,59.6,02:29,149
3,137188,Weight,59.6,02:44,164
4,137188,Weight,59.6,02:59,179
5,137188,Weight,59.6,03:14,194
6,137188,Weight,59.6,03:29,209
7,137188,Weight,59.6,03:59,239
8,137188,Weight,59.6,04:29,269
9,137188,Weight,59.6,05:29,329


In [90]:
def compute_urine_rate_slope(df_sub):

    if len(df_sub) < 2:
        return 0.0

    df_sub = df_sub.sort_values('Time_Minute').reset_index(drop=True)

    times_hrs = df_sub['Time_Minute'].values / 60.0
    volumes = df_sub['Value'].values
    time_deltas = np.diff(times_hrs, prepend=0)
    time_deltas = np.where(time_deltas == 0, 0.016, time_deltas)
    rates = volumes / time_deltas
    slope, intercept, r_val, p_val, std_err = linregress(times_hrs, rates)

    return slope

In [91]:
compute_urine_rate_slope(df_sub)

np.float64(-2.3801396449976764)

In [92]:
def find_uminr(df_sub, df_sub2):
  TIME_STABILITY_THRESHOLD = 0.1

  if len(df_sub2) == 0:
    weight_mean = np.nan
    return np.nan
  elif len(df_sub2) == 1:
    if df_sub2.isnull().sum().sum() != 0:
      weight_mean = np.nan
      return np.nan
    else:
      weight_mean = df_sub2['Value'].iloc[0]
  else:
    weight_mean = df_sub2['Value'].mean()

  urine_delta = df_sub.groupby('Time_Minute')['Value'].max()
  time_delta = df_sub.groupby('Time_Minute')['Time_Minute'].max().diff()/60
  umr_series = pd.Series(np.where((time_delta > TIME_STABILITY_THRESHOLD) & (weight_mean > 0), urine_delta/(weight_mean*time_delta), np.nan))
  uminr = umr_series.min()
  return uminr

In [93]:
find_uminr(df_sub, df_sub2)

np.float64(0.16778523489932884)

In [94]:
def find_umaxr(df_sub, df_sub2):
  TIME_STABILITY_THRESHOLD = 0.1

  if len(df_sub2) == 0:
    weight_mean = np.nan
    return np.nan
  elif len(df_sub2) == 1:
    if df_sub2.isnull().sum().sum() != 0:
      weight_mean = np.nan
      return np.nan
    else:
      weight_mean = df_sub2['Value'].iloc[0]
  else:
    weight_mean = df_sub2['Value'].mean()

  urine_delta = df_sub.groupby('Time_Minute')['Value'].max()
  time_delta = df_sub.groupby('Time_Minute')['Time_Minute'].max().diff()/60
  umr_series = pd.Series(np.where((time_delta > TIME_STABILITY_THRESHOLD) & (weight_mean > 0), urine_delta/(weight_mean*time_delta), np.nan))
  umaxr = umr_series.max()
  return umaxr

In [95]:
find_umaxr(df_sub, df_sub2)

np.float64(3.7751677852348986)

In [96]:
def find_umeanr(df_sub, df_sub2):
  TIME_STABILITY_THRESHOLD = 0.1

  if len(df_sub2) == 0:
    weight_mean = np.nan
    return np.nan
  elif len(df_sub2) == 1:
    if df_sub2.isnull().sum().sum() != 0:
      weight_mean = np.nan
      return np.nan
    else:
      weight_mean = df_sub2['Value'].iloc[0]
  else:
    weight_mean = df_sub2['Value'].mean()

  urine_delta = df_sub.groupby('Time_Minute')['Value'].max()
  time_delta = df_sub.groupby('Time_Minute')['Time_Minute'].max().diff()/60
  umr_series = pd.Series(np.where((time_delta > TIME_STABILITY_THRESHOLD) & (weight_mean > 0), urine_delta/(weight_mean*time_delta), np.nan))
  umeanr = umr_series.mean()
  return umeanr

In [97]:
find_umeanr(df_sub, df_sub2)

np.float64(0.9345104932353255)

In [98]:
df_temp = dfs_filled[3]

In [99]:
df_sub = df_temp[df_temp['Parameter'] == 'DiasABP'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,136263,DiasABP,58.0,03:11,191
1,136263,DiasABP,58.0,03:26,206
2,136263,DiasABP,52.0,03:41,221
3,136263,DiasABP,56.0,03:56,236
4,136263,DiasABP,67.0,04:11,251
5,136263,DiasABP,66.0,04:26,266
6,136263,DiasABP,55.0,05:11,311
7,136263,DiasABP,67.0,05:41,341
8,136263,DiasABP,48.0,06:11,371
9,136263,DiasABP,59.0,06:41,401


In [100]:
df_sub2 = df_temp[df_temp['Parameter'] == 'NIDiasABP'].reset_index(drop=True)
df_sub2

,RecordID,Parameter,Value,Time,Time_Minute
0,136263,NIDiasABP,42.0,24:11,1451
1,136263,NIDiasABP,49.0,24:21,1461
2,136263,NIDiasABP,61.0,45:01,2701
3,136263,NIDiasABP,55.0,45:11,2711
4,136263,NIDiasABP,62.0,46:11,2771
5,136263,NIDiasABP,59.0,46:41,2801
6,136263,NIDiasABP,55.0,47:11,2831


In [101]:
def minutes_to_time_string(total_minutes):

  if pd.isna(total_minutes) or total_minutes is None or total_minutes < 0:
    return np.nan

  duration = timedelta(minutes=int(total_minutes))
  total_seconds = duration.total_seconds()
  total_hours = math.floor(total_seconds / 3600)
  remaining_seconds = total_seconds % 3600
  minutes = math.floor(remaining_seconds / 60)
  time_string = f"{int(total_hours):02d}:{int(minutes):02d}"

  return time_string

In [102]:
def find_uniabp(df_sub, df_sub2):

  i_series = df_sub.groupby('Time_Minute').agg({'Time_Minute': 'min', 'Value': 'min'}).set_index('Time_Minute').sort_index()
  ni_series = df_sub2.groupby('Time_Minute').agg({'Time_Minute': 'min', 'Value': 'min'}).set_index('Time_Minute').sort_index()

  match_index = i_series.index.intersection(ni_series.index)

  if len(match_index) == 0:
    delta = 0
  else:
    i_series_match = i_series.reindex(match_index)
    ni_series_match = ni_series.reindex(match_index)
    delta = 0

  uni_index = i_series.index.union(ni_series.index)

  uni_series_preindex = i_series.reindex(uni_index).fillna(ni_series)
  uni_series = uni_series_preindex.reset_index()
  df_uni = uni_series.rename(columns = {'index':'Time_Minute', 'value':'Value'})

  recordid_series = pd.Series(data=np.repeat(df_sub['RecordID'][0], len(df_uni)), name='RecordID').reset_index(drop=True)
  parameter_series = pd.Series(data=np.repeat('Uni'+df_sub['Parameter'][0], len(df_uni)), name='Parameter').reset_index(drop=True)
  value_series = round(df_uni['Value'], 2)
  time_minute_series = df_uni['Time_Minute']
  time_series = time_minute_series.apply(minutes_to_time_string)
  df_result = pd.concat([recordid_series, parameter_series, value_series, time_series, time_minute_series], axis=1)
  df_result.columns = ['RecordID', 'Parameter', 'Value', 'Time', 'Time_Minute']

  return df_result

In [103]:
def find_unidf(df_sub, df_sub2):
  if (len(df_sub) == 0) and (len(df_sub2) == 0):
    columns_list = ['RecordID', 'Parameter', 'Value', 'Time', 'Time_Minute']
    df_result = pd.DataFrame(columns=columns_list)
  elif len(df_sub) == 0:
    df_sub2['Parameter'] = 'Uni'+ df_sub2['Parameter'].str.replace('^NI', '', regex=True)
    df_result = df_sub2
  elif len(df_sub2) == 0:
    df_sub['Parameter'] = 'Uni'+df_sub['Parameter']
    df_result = df_sub
  else:
    df_result = find_uniabp(df_sub, df_sub2)

  return df_result


In [104]:
find_unidf(df_sub, df_sub2)

,RecordID,Parameter,Value,Time,Time_Minute
0,136263,UniDiasABP,58.0,03:11,191
1,136263,UniDiasABP,58.0,03:26,206
2,136263,UniDiasABP,52.0,03:41,221
3,136263,UniDiasABP,56.0,03:56,236
4,136263,UniDiasABP,67.0,04:11,251
5,136263,UniDiasABP,66.0,04:26,266
6,136263,UniDiasABP,55.0,05:11,311
7,136263,UniDiasABP,67.0,05:41,341
8,136263,UniDiasABP,48.0,06:11,371
9,136263,UniDiasABP,59.0,06:41,401


In [105]:
def mad_stats(df1, df2):
  combined_index = df1.index.union(df2.index)

  df1_imputed = df1.reindex(combined_index).ffill().bfill()
  df2_imputed = df2.reindex(combined_index).ffill().bfill()

  mad = (df1_imputed['Value'] - df2_imputed['Value']).abs().mean()

  return mad

In [106]:
mad_stats(df_sub, df_sub2)

np.float64(7.357142857142857)

In [107]:
def find_mad(df1, df2):
  if (len(df1) == 0) and (len(df2) == 0):
    return 0
  elif len(df1) < 2:
    return 0
  elif len(df2) < 2:
    return 0
  else:
    return mad_stats(df1, df2)

In [108]:
find_mad(df_sub, df_sub2)

np.float64(7.357142857142857)

In [109]:
df_temp = dfs_filled[4]

In [110]:
df_sub = df_temp[df_temp['Parameter'] == 'Albumin'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute


In [111]:
def compute_end_to_end_slope(df_sun):

  clean_data = df_sub.dropna(subset=['Value'])

  if len(clean_data) < 2:
    return 0.0

  v_initial = clean_data['Value'].iloc[0]
  v_final = clean_data['Value'].iloc[-1]

  t_initial = clean_data['Time_Minute'].iloc[0]
  t_final = clean_data['Time_Minute'].iloc[-1]
  t_diff = t_final - t_initial

  if t_diff == 0.0:
    return 0.0
  else:
    slope = (v_final - v_initial)/t_diff
    return slope

In [112]:
compute_end_to_end_slope(df_sub)

0.0

In [113]:
def compute_regression_slope(df_sub):
  clean_data = df_sub.dropna(subset=['Value'])

  if len(clean_data) < 2 or np.ptp(clean_data['Time_Minute']) == 0:
    return 0.0

  X = clean_data['Time_Minute'].values
  y = clean_data['Value'].values

  slope, intercept, r_value, p_value, std_err = linregress(X, y)
  return slope

In [114]:
df_sub = df_temp[df_temp['Parameter'] == 'GCS'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,141669,GCS,15.0,02:56,176
1,141669,GCS,15.0,05:56,356
2,141669,GCS,15.0,09:56,596
3,141669,GCS,15.0,13:56,836
4,141669,GCS,15.0,17:56,1076
5,141669,GCS,15.0,21:56,1316
6,141669,GCS,15.0,25:56,1556
7,141669,GCS,15.0,29:56,1796
8,141669,GCS,15.0,33:56,2036
9,141669,GCS,15.0,37:56,2276


In [115]:
compute_regression_slope(df_sub)

np.float64(0.0)

In [116]:
df_temp = dfs_filled[4]

In [117]:
df_sub = df_temp[df_temp['Parameter'] == 'TroponinI'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute


In [118]:
df_sub2 = df_temp[df_temp['Parameter'] == 'TroponinT'].reset_index(drop=True)
df_sub2

,RecordID,Parameter,Value,Time,Time_Minute


In [119]:
def combine_troponin(df_sub, df_sub2):
  i_series = df_sub.groupby('Time_Minute').agg({'Time_Minute': 'min', 'Value': 'max'}).set_index('Time_Minute').sort_index()
  t_series = df_sub2.groupby('Time_Minute').agg({'Time_Minute': 'min', 'Value': 'max'}).set_index('Time_Minute').sort_index()

  uni_index = i_series.index.union(t_series.index)

  uni_series_preindex = i_series.reindex(uni_index).fillna(t_series)
  uni_series = uni_series_preindex.reset_index()
  df_uni = uni_series.rename(columns = {'index':'Time_Minute', 'value':'Value'})

  recordid_series = pd.Series(data=np.repeat(df_sub['RecordID'][0], len(df_uni)), name='RecordID').reset_index(drop=True)
  parameter_series = pd.Series(data=np.repeat('Troponin', len(df_uni)), name='Parameter').reset_index(drop=True)
  value_series = df_uni['Value']
  time_minute_series = df_uni['Time_Minute']
  time_series = time_minute_series.apply(minutes_to_time_string)
  df_result = pd.concat([recordid_series, parameter_series, value_series, time_series, time_minute_series], axis=1)
  df_result.columns = ['RecordID', 'Parameter', 'Value', 'Time', 'Time_Minute']

  return df_result

In [120]:
def find_troponin(df_sub, df_sub2):
  if (len(df_sub) == 0) and (len(df_sub2) == 0):
    columns_list = ['RecordID', 'Parameter', 'Value', 'Time', 'Time_Minute']
    return pd.DataFrame(columns=columns_list)
  elif len(df_sub) == 0:
    df_sub2['Parameter'] = 'Troponin'
    return df_sub2
  elif len(df_sub2) == 0:
    df_sub['Parameter'] = 'Troponin'
    return df_sub
  else:
    df_result = combine_troponin(df_sub, df_sub2)
    return df_result

In [121]:
find_troponin(df_sub, df_sub2)

,RecordID,Parameter,Value,Time,Time_Minute


# icu data column

In [122]:
col_stats = ['Albumin_Initial', 'Albumin_Min', 'Albumin_Max', 'Albumin_Mean', 'ALP_Initial', 'ALP_Max', 'ALT_Initial', 'ALT_Max', 'AST_Initial', 'AST_Max',
'Bilirubin_Initial', 'Bilirubin_Max', 'BUN_Initial', 'BUN_Max', 'BUN_Mean', 'BUN_Creatinine_Ratio', 'Cholesterol_Initial', 'Cholesterol_Min',
'Creatinine_Initial', 'Creatinine_Max', 'Creatinine_Mean', 'FiO2_Initial', 'FiO2_Max', 'FiO2_Mean',
'GCS_Initial', 'GCS_Final', 'GCS_Min', 'GCS_Max', 'GCS_Mean', 'Glucose_Initial', 'Glucose_Min', 'Glucose_Max', 'Glucose_Mean',
'HCO3_Initial', 'HCO3_Final', 'HCO3_Min', 'HCO3_Max', 'HCT_Initial', 'HCT_Min', 'HCT_Max', 'HCT_Mean',
'HR_Initial', 'HR_Final', 'HR_Min', 'HR_Max', 'HR_Mean', 'HR_SD', 'K_Initial', 'K_Final', 'K_Min', 'K_Max', 'K_Mean',
'Lactate_Initial', 'Lactate_Max', 'Lactate_Clearance_Rate_6H', 'Lactate_Clearance_Rate_12H', 'MechVent_Any',
'Mg_Initial', 'Mg_Min', 'Mg_Max', 'Na_Initial', 'Na_Final', 'Na_Min', 'Na_Max', 'Na_Mean',
'PaCO2_Initial', 'PaCO2_Min', 'PaCO2_Max', 'PaO2_Initial', 'PaO2_Min', 'PaO2_FiO2_Ratio',
'pH_Initial', 'pH_Min', 'pH_Max', 'pH_Mean', 'Platelets_Initial', 'Platelets_Min',
'RespRate_Initial', 'RespRate_Max', 'RespRate_Mean', 'SaO2_Initial', 'SaO2_Min',
'Temp_Initial', 'Temp_Min', 'Temp_Max', 'Temp_Mean', 'Temp_SD',
'Urine_Sum', 'Urine_MinRate', 'Urine_MaxRate', 'Urine_MeanRate',
'WBC_Initial', 'WBC_Min', 'WBC_Max', 'Weight_Initial', 'Weight_Final',
'UniDiasABP_Initial', 'UniDiasABP_Min', 'UniDiasABP_Max', 'UniDiasABP_Mean',
'UniMAP_Initial', 'UniMAP_Final', 'UniMAP_Min', 'UniMAP_Max', 'UniMAP_Mean', 'UniMAP_SD',
'UniSysABP_Initial', 'UniSysABP_Final', 'UniSysABP_Min', 'UniSysABP_Max', 'UniSysABP_Mean', 'UniSysABP_SD', 'Troponin_Max']

In [123]:
col_count = ['Albumin_Count', 'ALP_Count', 'ALT_Count', 'AST_Count', 'BUN_Count', 'Bilirubin_Count',
'Cholesterol_Count', 'Creatinine_Count', 'FiO2_Count', 'GCS_Count', 'Glucose_Count',
'HCO3_Count', 'HCT_Count', 'HR_Count', 'K_Count', 'Lactate_Count', 'MechVent_Count',
'Mg_Count', 'Na_Count', 'PaCO2_Count', 'PaO2_Count', 'pH_Count', 'Platelets_Count',
'RespRate_Count', 'SaO2_Count', 'Temp_Count', 'Urine_Count', 'WBC_Count',
'UniDiasABP_Count', 'UniMAP_Count', 'UniSysABP_Count', 'Troponin_Count']

In [124]:
col_delta = ['Albumin_Delta', 'ALP_Delta', 'ALT_Delta', 'AST_Delta', 'Bilirubin_Delta', 'BUN_Delta', 'Cholesterol_Delta', 'Creatinine_Delta',
'HCO3_Delta', 'HCT_Delta', 'K_Delta', 'Mg_Delta', 'Na_Delta', 'PaO2_Delta', 'Platelets_Delta', 'SaO2_Delta', 'WBC_Delta', 'Weight_Delta', 'Troponin_Delta']

In [125]:
col_slope = ['FiO2_Slope', 'GCS_Slope', 'Glucose_Slope', 'HR_Slope', 'PaCO2_Slope', 'pH_Slope', 'RespRate_Slope', 'Temp_Slope', 'Urine_Rate_Slope', 'UniDiasABP_Slope', 'UniMAP_Slope', 'UniSysABP_Slope']

In [126]:
col_miss = ['Albumin_Miss', 'ALP_Miss', 'ALT_Miss', 'AST_Miss', 'Bilirubin_Miss', 'BUN_Miss', 'Cholesterol_Miss', 'Creatinine_Miss',
'FiO2_Miss', 'GCS_Miss', 'Glucose_Miss', 'HCO3_Miss', 'HCT_Miss', 'HR_Miss', 'K_Miss', 'Lactate_Miss', 'MechVent_Miss', 'Mg_Miss', 'Na_Miss',
'PaCO2_Miss', 'PaO2_Miss', 'pH_Miss', 'Platelets_Miss', 'RespRate_Miss', 'SaO2_Miss', 'Temp_Miss', 'Urine_Miss', 'WBC_Miss', 'Weight_Miss',
'UniDiasABP_Miss', 'UniMAP_Miss', 'UniSysABP_Miss', 'Troponin_Miss']

In [127]:
col_last_minute = ['Albumin_Last_Minute', 'ALP_Last_Minute', 'ALT_Last_Minute', 'AST_Last_Minute', 'Bilirubin_Last_Minute', 'BUN_Last_Minute', 'Cholesterol_Last_Minute', 'Creatinine_Last_Minute',
'FiO2_Last_Minute', 'GCS_Last_Minute', 'Glucose_Last_Minute', 'HCO3_Last_Minute', 'HCT_Last_Minute', 'HR_Last_Minute', 'K_Last_Minute', 'Lactate_Last_Minute',
'MechVent_Last_Minute', 'Mg_Last_Minute', 'Na_Last_Minute', 'PaCO2_Last_Minute', 'PaO2_Last_Minute', 'pH_Last_Minute', 'Platelets_Last_Minute', 'RespRate_Last_Minute',
'SaO2_Last_Minute', 'Temp_Last_Minute', 'Urine_Last_Minute', 'WBC_Last_Minute', 'Weight_Last_Minute', 'UniDiasABP_Last_Minute', 'UniMAP_Last_Minute', 'UniSysABP_Last_Minute', 'Troponin_Last_Minute']

In [128]:
col_df = ['RecordID'] + col_stats + col_count + col_delta + col_slope + col_miss + col_last_minute

In [129]:
df_record = pd.DataFrame(columns=col_df)
df_record.loc[0] = np.nan
df_record['RecordID'] = df_temp['RecordID'].iloc[0]

In [130]:
df_record

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute
0,141669,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

# icu data function

In [131]:
df_temp = dfs_filled[0]

In [132]:
df_sub = df_temp[df_temp['Parameter'] == 'Albumin'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Albumin,2.3,03:22,202
1,137188,Albumin,2.2,11:50,710
2,137188,Albumin,2.3,35:13,2113


In [133]:
# Albumin
# Albumin_Initial
# Albumin_Min
# Albumin_Max
# Albumin_Mean
# Albumin_Count
# Albumin_Delta
# Albumin_Miss
# Albumin_Last_Minute


def albumin_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[ col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[ col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [134]:
albumin_stats(df_sub, df_record, 'Albumin')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [135]:
df_sub = df_temp[df_temp['Parameter'] == 'ALP'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,ALP,84.0,03:22,202
1,137188,ALP,64.0,11:50,710
2,137188,ALP,55.0,35:13,2113


In [136]:
# ALP
# ALP_Initial
# ALP_Max
# ALP_Count
# ALP_Delta
# ALP_Miss
# ALP_Last_Minute


def alp_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_max = col+'_Max'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_max] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [137]:
alp_stats(df_sub, df_record, 'ALP')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2113,2113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [138]:
df_sub = df_temp[df_temp['Parameter'] == 'ALT'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,ALT,20.0,03:22,202
1,137188,ALT,18.0,11:50,710
2,137188,ALT,14.0,35:13,2113


In [139]:
# ALT
# ALT_Initial
# ALT_Max
# ALT_Count
# ALT_Delta
# ALT_Miss
# ALT_Last_Minute


def alt_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_max = col+'_Max'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_max] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [140]:
alt_stats(df_sub, df_record, 'ALT')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2113,2113,2113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [141]:
df_sub = df_temp[df_temp['Parameter'] == 'AST'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,AST,123.0,03:22,202
1,137188,AST,97.0,11:50,710
2,137188,AST,70.0,35:13,2113


In [142]:
# AST
# AST_Initial
# AST_Max
# AST_Count
# AST_Delta
# AST_Miss
# AST_Last_Minute


def ast_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_max = col+'_Max'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_max] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [143]:
ast_stats(df_sub, df_record, 'AST')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2113,2113,2113,2113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [144]:
df_sub = df_temp[df_temp['Parameter'] == 'Bilirubin'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Bilirubin,6.6,03:22,202
1,137188,Bilirubin,7.6,11:50,710
2,137188,Bilirubin,11.2,35:13,2113


In [145]:
# Bilirubin
# Bilirubin_Initial
# Bilirubin_Max
# Bilirubin_Count
# Bilirubin_Delta
# Bilirubin_Miss
# Bilirubin_Last_Minute


def bilirubin_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_max = col+'_Max'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_max] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [146]:
bilirubin_stats(df_sub, df_record, 'Bilirubin')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2113,2113,2113,2113,2113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

In [147]:
df_sub = df_temp[df_temp['Parameter'] == 'BUN'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,BUN,7.0,03:22,202
1,137188,BUN,5.0,11:50,710
2,137188,BUN,4.0,22:00,1320
3,137188,BUN,6.0,35:13,2113


In [148]:
df_sub2 = df_temp[df_temp['Parameter'] == 'Creatinine'].reset_index(drop=True)
df_sub2

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Creatinine,0.9,03:22,202
1,137188,Creatinine,0.7,11:50,710
2,137188,Creatinine,0.5,22:00,1320
3,137188,Creatinine,0.6,35:13,2113


In [149]:
# BUN
# BUN_Initial
# BUN_Max
# BUN_Mean
# BUN_Count
# BUN_Delta
# BUN_Miss
# BUN_Creatinine_Ratio - Max BUN/Creatinine Ratio
# BUN_Last_Minute


def bun_stats(df_sub, df_sub2, df_record, col):
  col_initial = col+'_Initial'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_bcr = col+'_Creatinine_Ratio'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_bcr] = np.nan
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_bcr] = np.nan
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_bcr] = find_max_bcr(df_sub, df_sub2)
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_bcr] = np.nan
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_bcr] = find_max_bcr(df_sub, df_sub2)
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [150]:
bun_stats(df_sub, df_sub2, df_record, 'BUN')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2113,2113,2113,2113,2113,2113,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [151]:
df_sub = df_temp[df_temp['Parameter'] == 'Cholesterol'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute


In [152]:
# Cholesterol
# Cholesterol_Initial
# Cholesterol_Min
# Cholesterol_Count
# Cholesterol_Delta
# Cholesterol_Miss
# Cholesterol_Last_Minute


def cholesterol_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [153]:
cholesterol_stats(df_sub, df_record, 'Cholesterol')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2113,2113,2113,2113,2113,2113,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [154]:
df_sub = df_temp[df_temp['Parameter'] == 'Creatinine'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Creatinine,0.9,03:22,202
1,137188,Creatinine,0.7,11:50,710
2,137188,Creatinine,0.5,22:00,1320
3,137188,Creatinine,0.6,35:13,2113


In [155]:
# Creatinine
# Creatinine_Initial
# Creatinine_Max
# Creatinine_Mean
# Creatinine_Count
# Creatinine_Delta
# Creatinine_Miss
# Creatinine_Last_Minute


def creatinine_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [156]:
creatinine_stats(df_sub, df_record, 'Creatinine')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2113,2113,2113,2113,2113,2113,0,2113,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [157]:
df_sub = df_temp[df_temp['Parameter'] == 'DiasABP'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,DiasABP,65.0,17:59,1079
1,137188,DiasABP,71.0,18:29,1109
2,137188,DiasABP,69.0,18:59,1139
3,137188,DiasABP,63.0,19:29,1169
4,137188,DiasABP,59.0,19:44,1184
5,137188,DiasABP,46.0,19:54,1194
6,137188,DiasABP,45.0,19:59,1199
7,137188,DiasABP,62.0,20:29,1229
8,137188,DiasABP,60.0,21:29,1289
9,137188,DiasABP,66.0,22:29,1349


In [158]:
# DiasABP
# DiasABP_Initial
# DiasABP_Min
# DiasABP_Max
# DiasABP_Mean
# DiasABP_SD
# DiasABP_Time_Duration - threshold < 40 x
# DiasABP_Count
# DiasABP_Slope
# DiasABP_Miss
# DiasABP_Last_Minute


def diasabp_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_sd = col+'_SD'
  #col_td = col+'_Time_Duration'
  #threshold = 40
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_sd] = np.nan
    #df_record[col_td] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = 0
      #df_record[col_td] = 0
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = df_sub['Value'].std()
      #max_duration, max_begin, max_end = find_max_duration(df_sub, threshold, 'below')
      #time_duration, time_begin, time_end = find_time_duration(df_sub, max_begin, max_end)
      #df_record[col_td] = time_duration
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [159]:
diasabp_stats(df_sub, df_record, 'DiasABP')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [160]:
df_sub = df_temp[df_temp['Parameter'] == 'FiO2'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,FiO2,1.0,02:04,124
1,137188,FiO2,1.0,05:47,347
2,137188,FiO2,0.8,08:34,514
3,137188,FiO2,0.8,11:43,703
4,137188,FiO2,0.7,14:29,869
5,137188,FiO2,0.7,18:44,1124
6,137188,FiO2,0.7,20:09,1209
7,137188,FiO2,0.7,23:10,1390
8,137188,FiO2,0.7,27:29,1649
9,137188,FiO2,0.6,28:29,1709


In [161]:
# FiO2
# FiO2_Initial
# FiO2_Max
# FiO2_Mean
# FiO2_Count
# FiO2_Slope
# FiO2_Miss
# FiO2_Last_Minute


def fio2_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [162]:
fio2_stats(df_sub, df_record, 'FiO2')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000198,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [163]:
df_sub = df_temp[df_temp['Parameter'] == 'GCS'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,GCS,3.0,02:14,134
1,137188,GCS,6.0,04:29,269
2,137188,GCS,6.0,06:29,389
3,137188,GCS,7.0,10:29,629
4,137188,GCS,7.0,14:29,869
5,137188,GCS,7.0,18:29,1109
6,137188,GCS,4.0,22:29,1349
7,137188,GCS,4.0,26:29,1589
8,137188,GCS,7.0,30:29,1829
9,137188,GCS,4.0,34:29,2069


In [164]:
# GCS
# GCS_Initial
# GCS_Final
# GCS_Min
# GCS_Max
# GCS_Mean
# GCS_Count
# GCS_Slope
# GCS_Miss
# GCS_Last_Minute


def gcs_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_final = col+'_Final'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_final] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [165]:
gcs_stats(df_sub, df_record, 'GCS')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,18,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000198,0.000168,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,0,0,NaN,NaN,NaN,NaN,

In [166]:
df_sub = df_temp[df_temp['Parameter'] == 'Glucose'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Glucose,101.0,03:22,202
1,137188,Glucose,88.0,11:50,710
2,137188,Glucose,148.0,22:00,1320
3,137188,Glucose,173.0,35:13,2113


In [167]:
# Glucose
# Glucose_Initial
# Glucose_Min
# Glucose_Max
# Glucose_Mean
# Glucose_Count
# Glucose_Slope
# Glucose_Miss
# Glucose_Last_Minute


def glucose_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [168]:
glucose_stats(df_sub, df_record, 'Glucose')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,18,13,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000198,0.000168,0.044188,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,0,0,0,NaN,

In [169]:
df_sub = df_temp[df_temp['Parameter'] == 'HCO3'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,HCO3,22.0,03:22,202
1,137188,HCO3,22.0,11:50,710
2,137188,HCO3,24.0,22:00,1320
3,137188,HCO3,25.0,35:13,2113


In [170]:
# HCO3
# HCO3_Initial
# HCO3_Final
# HCO3_Min
# HCO3_Max
# HCO3_Count
# HCO3_Delta
# HCO3_Miss
# HCO3_Last_Minute


def hco3_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_final = col+'_Final'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_final] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [171]:
hco3_stats(df_sub, df_record, 'HCO3')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,18,13,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000198,0.000168,0.044188,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,0,0,0,0,

In [172]:
df_sub = df_temp[df_temp['Parameter'] == 'HCT'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,HCT,29.7,03:22,202
1,137188,HCT,26.7,11:50,710
2,137188,HCT,29.7,22:00,1320
3,137188,HCT,28.8,27:18,1638
4,137188,HCT,29.0,35:13,2113
5,137188,HCT,26.6,46:39,2799


In [173]:
# HCT
# HCT_Initial
# HCT_Min
# HCT_Max
# HCT_Mean
# HCT_Count
# HCT_Delta
# HCT_Miss
# HCT_Last_Minute


def hct_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [174]:
hct_stats(df_sub, df_record, 'HCT')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,18,13,4,4,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,3.0,-3.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000198,0.000168,0.044188,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,

In [175]:
df_sub = df_temp[df_temp['Parameter'] == 'HR'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,HR,109.0,02:14,134
1,137188,HR,109.0,02:29,149
2,137188,HR,111.0,02:44,164
3,137188,HR,113.0,02:59,179
4,137188,HR,113.0,03:14,194
5,137188,HR,113.0,03:29,209
6,137188,HR,124.0,03:59,239
7,137188,HR,113.0,04:29,269
8,137188,HR,116.0,05:29,329
9,137188,HR,111.0,06:29,389


In [176]:
# HR
# HR_Initial
# HR_Final
# HR_Min
# HR_Max
# HR_Mean
# HR_SD
# HR_Count
# HR_Slope
# HR_Miss
# HR_Last_Minute


def hr_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_final = col+'_Final'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_sd = col+'_SD'
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_final] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_sd] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = 0
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = df_sub['Value'].std()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [177]:
hr_stats(df_sub, df_record, 'HR')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,18,13,4,4,6,64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,3.0,-3.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000198,0.000168,0.044188,-0.010776,NaN,NaN,NaN,NaN,NaN,NaN

In [178]:
df_sub = df_temp[df_temp['Parameter'] == 'K'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,K,4.0,03:22,202
1,137188,K,4.0,03:32,212
2,137188,K,3.6,11:50,710
3,137188,K,4.1,22:00,1320
4,137188,K,4.6,35:13,2113


In [179]:
# K
# K_Initial
# K_Final
# K_Min
# K_Max
# K_Mean
# K_Count
# K_Delta
# K_Miss
# K_Last_Minute


def k_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_final = col+'_Final'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_final] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [180]:
k_stats(df_sub, df_record, 'K')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,18,13,4,4,6,64,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,3.0,-3.1,0.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000198,0.000168,0.044188,-0.010776,NaN,NaN,NaN,NaN,NaN,NaN,

In [181]:
df_sub = df_temp[df_temp['Parameter'] == 'Lactate'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Lactate,1.5,07:35,455
1,137188,Lactate,1.5,09:33,573
2,137188,Lactate,1.8,15:08,908
3,137188,Lactate,1.4,22:18,1338
4,137188,Lactate,1.4,35:50,2150
5,137188,Lactate,1.4,46:43,2803


In [182]:
# Lactate
# Lactate_Initial
# Lactate_Max
# Lactate_Count
# Lactate_Miss
# Lactate_Clearance_Rate_6H - Clearance Rate (% reduction over 6h)
# Lactate_Clearance_Rate_12H - Clearance Rate (% reduction over 12h)
# Lactate_Last_Minute


def lactate_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_max = col+'_Max'
  col_count = col+'_Count'
  col_miss = col+'_Miss'
  col_cr6 = col+'_Clearance_Rate_6H'
  col_cr12 = col+'_Clearance_Rate_12H'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_max] = np.nan
    df_record[col_count] = 0
    df_record[col_miss] = 1
    df_record[col_cr6] = np.nan
    df_record[col_cr12] = np.nan
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_miss] = 1
      df_record[col_cr6] = np.nan
      df_record[col_cr12] = np.nan
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_miss] = 0
      df_record[col_cr6] = np.nan
      df_record[col_cr12] = np.nan
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_miss] = 1
      df_record[col_cr6] = np.nan
      df_record[col_cr12] = np.nan
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_miss] = 0
      df_record[col_cr6] = find_lcr(df_sub, 6)
      df_record[col_cr12] = find_lcr(df_sub, 12)
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [183]:
lactate_stats(df_sub, df_record, 'Lactate')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,18,13,4,4,6,64,5,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,3.0,-3.1,0.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000198,0.000168,0.044188,-0.010776,NaN,NaN,NaN,NaN,NaN,NaN,

In [184]:
df_sub = df_temp[df_temp['Parameter'] == 'MAP'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,MAP,81.0,17:59,1079
1,137188,MAP,85.0,18:29,1109
2,137188,MAP,87.0,18:59,1139
3,137188,MAP,80.0,19:29,1169
4,137188,MAP,76.0,19:44,1184
5,137188,MAP,64.0,19:54,1194
6,137188,MAP,65.0,19:59,1199
7,137188,MAP,75.0,20:29,1229
8,137188,MAP,75.0,21:29,1289
9,137188,MAP,84.0,22:29,1349


In [185]:
# MAP
# MAP_Initial
# MAP_Min
# MAP_Max
# MAP_Mean
# MAP_SD
# MAP_Time_Duration - threshold < 65 x
# MAP_Count
# MAP_Slope
# MAP_Miss
# MAP_Last_Minute


def map_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_sd = col+'_SD'
  #col_td = col+'_Time_Duration'
  #threshold = 65
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_sd] = np.nan
    #df_record[col_td] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = 0
      #df_record[col_td] = 0
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = df_sub['Value'].std()
      #max_duration, max_begin, max_end = find_max_duration(df_sub, threshold, 'below')
      #time_duration, time_begin, time_end = find_time_duration(df_sub, max_begin, max_end)
      #df_record[col_td] = time_duration
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [186]:
map_stats(df_sub,df_record, 'MAP')

/tmp/ipykernel_245953/591698936.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_initial] = df_sub['Value'].iloc[0]
/tmp/ipykernel_245953/591698936.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_min] = df_sub['Value'].min()
/tmp/ipykernel_245953/591698936.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-frag

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,18,13,4,4,6,64,5,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,3.0,-3.1,0.6,NaN,N

In [187]:
df_sub = df_temp[df_temp['Parameter'] == 'MechVent'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,MechVent,1.0,02:04,124
1,137188,MechVent,1.0,05:47,347
2,137188,MechVent,1.0,06:29,389
3,137188,MechVent,1.0,10:29,629
4,137188,MechVent,1.0,11:43,703
5,137188,MechVent,1.0,14:29,869
6,137188,MechVent,1.0,18:44,1124
7,137188,MechVent,1.0,20:09,1209
8,137188,MechVent,1.0,23:10,1390
9,137188,MechVent,1.0,26:29,1589


In [188]:
# MechVent
# MechVent_Any - 1 if ventilated at any point
# MechVent_Count
# MechVent_Miss
# MechVent_Last_Minute


def mechvent_stats(df_sub, df_record, col):
  col_any = col+'_Any'
  col_count = col+'_Count'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_any] = np.nan
    df_record[col_count] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_any] = np.nan
      df_record[col_count] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_any] = (df_sub['Value'].iloc[0] == 1).any().astype(float)
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_any] = np.nan
      df_record[col_count] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_any] = (df_sub['Value'] == 1).any().astype(float)
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [189]:
mechvent_stats(df_sub,df_record, 'MechVent')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,18,13,4,4,6,64,5,6,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,3.0,-3.1,0.6,NaN,Na

In [190]:
df_sub = df_temp[df_temp['Parameter'] == 'Mg'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Mg,2.2,03:22,202
1,137188,Mg,1.7,11:50,710
2,137188,Mg,2.2,22:00,1320
3,137188,Mg,2.0,35:13,2113


In [191]:
# Mg
# Mg_Initial
# Mg_Min
# Mg_Max
# Mg_Count
# Mg_Delta
# Mg_Miss
# Mg_Last_Minute


def mg_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [192]:
mg_stats(df_sub, df_record, 'Mg')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.0,1.0,2.2,1.7,2.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,18,13,4,4,6,64,5,6,18,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,3.0,-3.1,0.6,-0.2,NaN

In [193]:
df_sub = df_temp[df_temp['Parameter'] == 'Na'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Na,136.0,03:22,202
1,137188,Na,139.0,11:50,710
2,137188,Na,136.0,22:00,1320
3,137188,Na,136.0,35:13,2113


In [194]:
# Na
# Na_Initial
# Na_Final
# Na_Min
# Na_Max
# Na_Mean
# Na_Count
# Na_Delta
# Na_Miss
# Na_Last_Minute


def na_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_final = col+'_Final'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_final] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [195]:
na_stats(df_sub,df_record, 'Na')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.0,1.0,2.2,1.7,2.2,136.0,136.0,136.0,139.0,136.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,3,4,3,0,4,18,13,4,4,6,64,5,6,18,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-29.0,-6.0,-53.0,4.6,-1.0,0,-0.3,3.0,-3.1,0.6

In [196]:
df_sub = df_temp[df_temp['Parameter'] == 'NIDiasABP'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,NIDiasABP,60.0,02:14,134
1,137188,NIDiasABP,62.0,02:29,149
2,137188,NIDiasABP,65.0,02:44,164
3,137188,NIDiasABP,70.0,02:59,179
4,137188,NIDiasABP,72.0,03:14,194
5,137188,NIDiasABP,74.0,03:29,209
6,137188,NIDiasABP,66.0,03:59,239
7,137188,NIDiasABP,66.0,04:29,269
8,137188,NIDiasABP,64.0,05:29,329
9,137188,NIDiasABP,65.0,06:29,389


In [197]:
# NIDiasABP
# NIDiasABP_Initial
# NIDiasABP_Min
# NIDiasABP_Max
# NIDiasABP_Mean
# NIDiasABP_SD
# NIDiasABP_Time_Duration - threshold < 40 x
# NIDiasABP_Count
# NIDiasABP_Slope
# NIDiasABP_Miss
# NIDiasABP_Last_Minute


def nidiasabp_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_sd = col+'_SD'
  #col_td = col+'_Time_Duration'
  #threshold = 40
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_sd] = np.nan
    #df_record[col_td] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = 0
      #df_record[col_td] = 0
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = df_sub['Value'].std()
      #max_duration, max_begin, max_end = find_max_duration(df_sub, threshold, 'below')
      #time_duration, time_begin, time_end = find_time_duration(df_sub, max_begin, max_end)
      #df_record[col_td] = time_duration
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [198]:
nidiasabp_stats(df_sub,df_record, 'NIDiasABP')

/tmp/ipykernel_245953/2456799477.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_initial] = df_sub['Value'].iloc[0]
/tmp/ipykernel_245953/2456799477.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_min] = df_sub['Value'].min()
/tmp/ipykernel_245953/2456799477.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-f

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.0,1.0,2.2,1.7,2.2,136.0,136.0,136.0,139.0,136.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [199]:
df_sub = df_temp[df_temp['Parameter'] == 'NIMAP'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,NIMAP,69.0,02:14,134
1,137188,NIMAP,70.0,02:29,149
2,137188,NIMAP,75.0,02:44,164
3,137188,NIMAP,80.0,02:59,179
4,137188,NIMAP,81.0,03:14,194
5,137188,NIMAP,83.0,03:29,209
6,137188,NIMAP,76.0,03:59,239
7,137188,NIMAP,76.0,04:29,269
8,137188,NIMAP,76.0,05:29,329
9,137188,NIMAP,75.0,06:29,389


In [200]:
# NIMAP
# NIMAP_Initial
# NIMAP_Min
# NIMAP_Max
# NIMAP_Mean
# NIMAP_SD
# NIMAP_Time_Duration - threshold < 65 x
# NIMAP_Count
# NIMAP_Slope
# NIMAP_Miss
# NIMAP_Last_Minute


def nimap_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_sd = col+'_SD'
  #col_td = col+'_Time_Duration'
  #threshold = 65
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_sd] = np.nan
    #df_record[col_td] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = 0
      #df_record[col_td] = 0
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = df_sub['Value'].std()
      #max_duration, max_begin, max_end = find_max_duration(df_sub, threshold, 'below')
      #time_duration, time_begin, time_end = find_time_duration(df_sub, max_begin, max_end)
      #df_record[col_td] = time_duration
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [201]:
nimap_stats(df_sub, df_record, 'NIMAP')

/tmp/ipykernel_245953/2795769658.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_initial] = df_sub['Value'].iloc[0]
/tmp/ipykernel_245953/2795769658.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_min] = df_sub['Value'].min()
/tmp/ipykernel_245953/2795769658.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-f

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.0,1.0,2.2,1.7,2.2,136.0,136.0,136.0,139.0,136.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [202]:
df_sub = df_temp[df_temp['Parameter'] == 'NISysABP'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,NISysABP,97.0,02:14,134
1,137188,NISysABP,98.0,02:29,149
2,137188,NISysABP,107.0,02:44,164
3,137188,NISysABP,112.0,02:59,179
4,137188,NISysABP,111.0,03:14,194
5,137188,NISysABP,111.0,03:29,209
6,137188,NISysABP,106.0,03:59,239
7,137188,NISysABP,108.0,04:29,269
8,137188,NISysABP,111.0,05:29,329
9,137188,NISysABP,114.0,06:29,389


In [203]:
# NISysABP
# NISysABP_Initial
# NISysABP_Min
# NISysABP_Max
# NISysABP_Mean
# NISysABP_SD
# NISysABP_Time_Duration - threshold < 90 x
# NISysABP_Count
# NISysABP_Slope
# NISysABP_Miss
# NISysABP_Last_Minute


def nisysabp_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_sd = col+'_SD'
  #col_td = col+'_Time_Duration'
  #threshold = 90
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_sd] = np.nan
    #df_record[col_td] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = 0
      #df_record[col_td] = 0
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = df_sub['Value'].std()
      #max_duration, max_begin, max_end = find_max_duration(df_sub, threshold, 'below')
      #time_duration, time_begin, time_end = find_time_duration(df_sub, max_begin, max_end)
      #df_record[col_td] = time_duration
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [204]:
nisysabp_stats(df_sub, df_record, 'NISysABP')

/tmp/ipykernel_245953/1988834833.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_initial] = df_sub['Value'].iloc[0]
/tmp/ipykernel_245953/1988834833.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_min] = df_sub['Value'].min()
/tmp/ipykernel_245953/1988834833.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-f

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.

In [205]:
df_sub = df_temp[df_temp['Parameter'] == 'PaCO2'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,PaCO2,40.0,08:33,513
1,137188,PaCO2,41.0,14:08,848
2,137188,PaCO2,44.0,21:18,1278
3,137188,PaCO2,47.0,26:49,1609
4,137188,PaCO2,51.0,34:50,2090
5,137188,PaCO2,39.0,36:46,2206
6,137188,PaCO2,40.0,45:43,2743


In [206]:
# PaCO2
# PaCO2_Initial
# PaCO2_Min
# PaCO2_Max
# PaCO2_Count
# PaCO2_Slope
# PaCO2_Miss
# PaCO2_Last_Minute


def paco2_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [207]:
paco2_stats(df_sub, df_record, 'PaCO2')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.

In [208]:
df_sub = df_temp[df_temp['Parameter'] == 'PaO2'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,PaO2,187.0,08:33,513
1,137188,PaO2,142.0,14:08,848
2,137188,PaO2,111.0,21:18,1278
3,137188,PaO2,115.0,26:49,1609
4,137188,PaO2,73.0,34:50,2090
5,137188,PaO2,160.0,36:46,2206
6,137188,PaO2,137.0,45:43,2743


In [209]:
df_sub2 = df_temp[df_temp['Parameter'] == 'FiO2'].reset_index(drop=True)
df_sub2

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,FiO2,1.0,02:04,124
1,137188,FiO2,1.0,05:47,347
2,137188,FiO2,0.8,08:34,514
3,137188,FiO2,0.8,11:43,703
4,137188,FiO2,0.7,14:29,869
5,137188,FiO2,0.7,18:44,1124
6,137188,FiO2,0.7,20:09,1209
7,137188,FiO2,0.7,23:10,1390
8,137188,FiO2,0.7,27:29,1649
9,137188,FiO2,0.6,28:29,1709


In [210]:
# PaO2
# PaO2_Initial
# PaO2_Min
# PaO2_Count
# PaO2_Delta
# PaO2_Miss
# PaO2_FiO2_Ratio - Min PaO2/FiO2 ​Ratio
# PaO2_Last_Minute


def pao2_stats(df_sub, df_sub2, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_pfr = col+'_FiO2_Ratio'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_pfr] = np.nan
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_pfr] = np.nan
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_pfr] = find_min_pfr(df_sub, df_sub2)
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_pfr] = np.nan
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_pfr] = find_min_pfr(df_sub, df_sub2)
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [211]:
pao2_stats(df_sub, df_sub2, df_record, 'PaO2')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.

In [212]:
df_sub = df_temp[df_temp['Parameter'] == 'pH'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,pH,7.39,08:33,513
1,137188,pH,7.37,14:08,848
2,137188,pH,7.34,21:18,1278
3,137188,pH,7.35,26:49,1609
4,137188,pH,7.29,34:50,2090
5,137188,pH,7.37,36:46,2206
6,137188,pH,7.42,45:43,2743


In [213]:
# pH
# pH_Initial
# pH_Min
# pH_Max
# pH_Mean
# pH_Count
# pH_Slope
# pH_Miss
# pH_Last_Minute


def ph_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [214]:
ph_stats(df_sub, df_record, 'pH')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.

In [215]:
df_sub = df_temp[df_temp['Parameter'] == 'Platelets'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Platelets,85.0,03:22,202
1,137188,Platelets,71.0,11:50,710
2,137188,Platelets,88.0,35:13,2113
3,137188,Platelets,94.0,46:39,2799


In [216]:
# Platelets
# Platelets_Initial
# Platelets_Min
# Platelets_Count
# Platelets_Delta
# Platelets_Miss
# Platelets_Last_Minute


def platelets_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [217]:
platelets_stats(df_sub, df_record, 'Platelets')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.

In [218]:
df_sub = df_temp[df_temp['Parameter'] == 'RespRate'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute


In [219]:
# RespRate
# RespRate_Initial
# RespRate_Max
# RespRate_Mean
# RespRate_Count
# RespRate_Slope
# RespRate_Miss
# RespRate_Last_Minute


def resprate_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [220]:
resprate_stats(df_sub, df_record, 'RespRate')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.

In [221]:
df_sub = df_temp[df_temp['Parameter'] == 'SaO2'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute


In [222]:
# SaO2
# SaO2_Initial
# SaO2_Min
# SaO2_Count
# SaO2_Delta
# SaO2_Miss
# SaO2_Last_Minute


def sao2_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [223]:
sao2_stats(df_sub, df_record, 'SaO2')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.06,1.5,1.8,0.0,-20.

In [224]:
df_sub = df_temp[df_temp['Parameter'] == 'SysABP'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,SysABP,112.0,17:59,1079
1,137188,SysABP,112.0,18:29,1109
2,137188,SysABP,124.0,18:59,1139
3,137188,SysABP,115.0,19:29,1169
4,137188,SysABP,107.0,19:44,1184
5,137188,SysABP,92.0,19:54,1194
6,137188,SysABP,96.0,19:59,1199
7,137188,SysABP,102.0,20:29,1229
8,137188,SysABP,103.0,21:29,1289
9,137188,SysABP,121.0,22:29,1349


In [225]:
# SysABP
# SysABP_Initial
# SysABP_Min
# SysABP_Max
# SysABP_Mean
# SysABP_SD
# SysABP_Time_Duration - threshold < 90 x
# SysABP_Count
# SysABP_Slope
# SysABP_Miss
# SysABP_Last_Minute


def sysabp_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_sd = col+'_SD'
  #col_td = col+'_Time_Duration'
  #threshold = 90
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_sd] = np.nan
    #df_record[col_td] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = 0
      #df_record[col_td] = 0
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = df_sub['Value'].std()
      #max_duration, max_begin, max_end = find_max_duration(df_sub, threshold, 'below')
      #time_duration, time_begin, time_end = find_time_duration(df_sub, max_begin, max_end)
      #df_record[col_td] = time_duration
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [226]:
sysabp_stats(df_sub, df_record, 'SysABP')

/tmp/ipykernel_245953/2051913423.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_initial] = df_sub['Value'].iloc[0]
/tmp/ipykernel_245953/2051913423.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_min] = df_sub['Value'].min()
/tmp/ipykernel_245953/2051913423.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-f

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute,SysABP_Initial,SysABP_Min,SysABP_Max,SysABP_Mean,SysABP_SD,SysABP_Count,SysABP_Slope,SysABP_Miss,SysABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,

In [227]:
df_sub = df_temp[df_temp['Parameter'] == 'Temp'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Temp,37.6,02:14,134
1,137188,Temp,37.6,04:29,269
2,137188,Temp,37.7,06:29,389
3,137188,Temp,38.7,10:29,629
4,137188,Temp,38.9,12:29,749
5,137188,Temp,38.8,13:29,809
6,137188,Temp,38.8,14:29,869
7,137188,Temp,37.7,16:14,974
8,137188,Temp,37.7,16:29,989
9,137188,Temp,37.4,16:44,1004


In [228]:
# Temp
# Temp_Initial
# Temp_Min
# Temp_Max
# Temp_Mean
# Temp_SD
# Temp_Count
# Temp_Slope
# Temp_Miss
# Temp_Last_Minute


def temp_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_sd = col+'_SD'
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_sd] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = 0
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = df_sub['Value'].std()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  return df_record

In [229]:
temp_stats(df_sub, df_record, 'Temp')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute,SysABP_Initial,SysABP_Min,SysABP_Max,SysABP_Mean,SysABP_SD,SysABP_Count,SysABP_Slope,SysABP_Miss,SysABP_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.9,0.675,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.5,22.0,25.0,

In [230]:
df_sub = df_temp[df_temp['Parameter'] == 'TroponinI'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute


In [231]:
# TroponinI
# TroponinI_Max
# TroponinI_Count
# TroponinI_Slope
# TroponinI_Miss
# TroponinI_Last_Minute


def troponini_stats(df_sub, df_record, col):
  col_max = col+'_Max'
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_max] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_end_to_end_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_end_to_end_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [232]:
troponini_stats(df_sub, df_record, 'TroponinI')

/tmp/ipykernel_245953/1837484740.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_max] = np.nan
/tmp/ipykernel_245953/1837484740.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_count] = 0
/tmp/ipykernel_245953/1837484740.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute,SysABP_Initial,SysABP_Min,SysABP_Max,SysABP_Mean,SysABP_SD,SysABP_Count,SysABP_Slope,SysABP_Miss,SysABP_Last_Minute,TroponinI_Max,TroponinI_Count,TroponinI_Slope,TroponinI_Miss,TroponinI_Last_Minute
0,141669,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.5,10.0,NaN,NaN,0.9,0.

In [233]:
df_sub = df_temp[df_temp['Parameter'] == 'TroponinT'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,TroponinT,0.40,03:22,202
1,137188,TroponinT,0.32,11:50,710
2,137188,TroponinT,0.22,22:00,1320


In [234]:
# TroponinT
# TroponinT_Max
# TroponinT_Count
# TroponinT_Slope
# TroponinT_Miss
# TroponinT_Last_Minute


def troponint_stats(df_sub, df_record, col):
  col_max = col+'_Max'
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_max] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_end_to_end_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_end_to_end_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [235]:
troponint_stats(df_sub, df_record, 'TroponinT')

/tmp/ipykernel_245953/2252501039.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_max] = df_sub['Value'].max()
/tmp/ipykernel_245953/2252501039.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_record[col_count] = df_sub['Value'].count()
/tmp/ipykernel_245953/2252501039.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fra

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute,SysABP_Initial,SysABP_Min,SysABP_Max,SysABP_Mean,SysABP_SD,SysABP_Count,SysABP_Slope,SysABP_Miss,SysABP_Last_Minute,TroponinI_Max,TroponinI_Count,TroponinI_Slope,TroponinI_Miss,TroponinI_Last_Minute,TroponinT_Max,TroponinT_Count,TroponinT_Slope,TroponinT_Miss,TroponinT_Last_Minute
0,141669,2.3,2.2,2.

In [236]:
df_sub = df_temp[df_temp['Parameter'] == 'Urine'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Urine,200.0,02:14,134
1,137188,Urine,380.0,04:29,269
2,137188,Urine,225.0,05:29,329
3,137188,Urine,150.0,06:29,389
4,137188,Urine,125.0,08:29,509
5,137188,Urine,100.0,10:29,629
6,137188,Urine,80.0,12:29,749
7,137188,Urine,75.0,14:29,869
8,137188,Urine,175.0,16:29,989
9,137188,Urine,125.0,17:29,1049


In [237]:
df_sub2 = df_temp[df_temp['Parameter'] == 'Weight'].reset_index(drop=True)
df_sub2

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Weight,59.6,00:00,0
1,137188,Weight,59.6,02:14,134
2,137188,Weight,59.6,02:29,149
3,137188,Weight,59.6,02:44,164
4,137188,Weight,59.6,02:59,179
5,137188,Weight,59.6,03:14,194
6,137188,Weight,59.6,03:29,209
7,137188,Weight,59.6,03:59,239
8,137188,Weight,59.6,04:29,269
9,137188,Weight,59.6,05:29,329


In [238]:
# Urine
# Urine_Sum = sum(x)
# Urine_MinRate - Min Rate (mL/kg/hr)
# Urine_MaxRate
# Urine_MeanRate
# Urine_Count
# Urine_Rate_Slope
# Urine_Miss
# Urine_Last_Minute


def urine_stats(df_sub, df_sub2, df_record, col):
  col_sum = col+'_Sum'
  col_minr = col+'_MinRate'
  col_maxr = col+'_MaxRate'
  col_meanr = col+'_MeanRate'
  col_count = col+'_Count'
  col_slope = col+'_Rate_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_sum] = np.nan
    df_record[col_minr] = np.nan
    df_record[col_maxr] = np.nan
    df_record[col_meanr] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_sum] = np.nan
      df_record[col_minr] = np.nan
      df_record[col_maxr] = np.nan
      df_record[col_meanr] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_sum] = df_sub['Value'].sum()
      df_record[col_minr] = find_uminr(df_sub, df_sub2)
      df_record[col_maxr] = find_umaxr(df_sub, df_sub2)
      df_record[col_meanr] = find_umeanr(df_sub, df_sub2)
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_urine_rate_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_sum] = np.nan
      df_record[col_minr] = np.nan
      df_record[col_maxr] = np.nan
      df_record[col_meanr] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_sum] = df_sub['Value'].sum()
      df_record[col_minr] = find_uminr(df_sub, df_sub2)
      df_record[col_maxr] = find_umaxr(df_sub, df_sub2)
      df_record[col_meanr] = find_umeanr(df_sub, df_sub2)
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_urine_rate_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  return df_record

In [239]:
urine_stats(df_sub, df_sub2, df_record, 'Urine')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute,SysABP_Initial,SysABP_Min,SysABP_Max,SysABP_Mean,SysABP_SD,SysABP_Count,SysABP_Slope,SysABP_Miss,SysABP_Last_Minute,TroponinI_Max,TroponinI_Count,TroponinI_Slope,TroponinI_Miss,TroponinI_Last_Minute,TroponinT_Max,TroponinT_Count,TroponinT_Slope,TroponinT_Miss,TroponinT_Last_Minute
0,141669,2.3,2.2,2.

In [240]:
df_sub = df_temp[df_temp['Parameter'] == 'WBC'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,WBC,13.3,03:22,202
1,137188,WBC,9.6,11:50,710
2,137188,WBC,10.4,35:13,2113
3,137188,WBC,9.3,46:39,2799


In [241]:
# WBC
# WBC_Initial
# WBC_Min
# WBC_Max
# WBC_Count
# WBC_Delta
# WBC_Miss
# WBC_Last_Minute


def wbc_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [242]:
wbc_stats(df_sub, df_record, 'WBC')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute,SysABP_Initial,SysABP_Min,SysABP_Max,SysABP_Mean,SysABP_SD,SysABP_Count,SysABP_Slope,SysABP_Miss,SysABP_Last_Minute,TroponinI_Max,TroponinI_Count,TroponinI_Slope,TroponinI_Miss,TroponinI_Last_Minute,TroponinT_Max,TroponinT_Count,TroponinT_Slope,TroponinT_Miss,TroponinT_Last_Minute
0,141669,2.3,2.2,2.

In [243]:
df_sub = df_temp[df_temp['Parameter'] == 'Weight'].reset_index(drop=True)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Weight,59.6,00:00,0
1,137188,Weight,59.6,02:14,134
2,137188,Weight,59.6,02:29,149
3,137188,Weight,59.6,02:44,164
4,137188,Weight,59.6,02:59,179
5,137188,Weight,59.6,03:14,194
6,137188,Weight,59.6,03:29,209
7,137188,Weight,59.6,03:59,239
8,137188,Weight,59.6,04:29,269
9,137188,Weight,59.6,05:29,329


In [244]:
# Weight
# Weight_Initial
# Weight_Final
# Weight_Count x
# Weight_Delta
# Weight_Miss
# Weight_Last_Minute


def weight_stats(df_sub, df_record, col):
  col_initial = col+'_Initial'
  col_final = col+'_Final'
  #col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_final] = np.nan
    #df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      #df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      #df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      #df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      #df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [245]:
weight_stats(df_sub, df_record, 'Weight')

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute,SysABP_Initial,SysABP_Min,SysABP_Max,SysABP_Mean,SysABP_SD,SysABP_Count,SysABP_Slope,SysABP_Miss,SysABP_Last_Minute,TroponinI_Max,TroponinI_Count,TroponinI_Slope,TroponinI_Miss,TroponinI_Last_Minute,TroponinT_Max,TroponinT_Count,TroponinT_Slope,TroponinT_Miss,TroponinT_Last_Minute
0,141669,2.3,2.2,2.

In [246]:
df1 = df_temp[df_temp['Parameter'] == 'DiasABP'].reset_index(drop=True)
df1

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,DiasABP,65.0,17:59,1079
1,137188,DiasABP,71.0,18:29,1109
2,137188,DiasABP,69.0,18:59,1139
3,137188,DiasABP,63.0,19:29,1169
4,137188,DiasABP,59.0,19:44,1184
5,137188,DiasABP,46.0,19:54,1194
6,137188,DiasABP,45.0,19:59,1199
7,137188,DiasABP,62.0,20:29,1229
8,137188,DiasABP,60.0,21:29,1289
9,137188,DiasABP,66.0,22:29,1349


In [247]:
df2 = df_temp[df_temp['Parameter'] == 'NIDiasABP'].reset_index(drop=True)
df2

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,NIDiasABP,60.0,02:14,134
1,137188,NIDiasABP,62.0,02:29,149
2,137188,NIDiasABP,65.0,02:44,164
3,137188,NIDiasABP,70.0,02:59,179
4,137188,NIDiasABP,72.0,03:14,194
5,137188,NIDiasABP,74.0,03:29,209
6,137188,NIDiasABP,66.0,03:59,239
7,137188,NIDiasABP,66.0,04:29,269
8,137188,NIDiasABP,64.0,05:29,329
9,137188,NIDiasABP,65.0,06:29,389


In [248]:
df_sub = find_unidf(df1, df2)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,UniDiasABP,60.0,02:14,134
1,137188,UniDiasABP,62.0,02:29,149
2,137188,UniDiasABP,65.0,02:44,164
3,137188,UniDiasABP,70.0,02:59,179
4,137188,UniDiasABP,72.0,03:14,194
5,137188,UniDiasABP,74.0,03:29,209
6,137188,UniDiasABP,66.0,03:59,239
7,137188,UniDiasABP,66.0,04:29,269
8,137188,UniDiasABP,64.0,05:29,329
9,137188,UniDiasABP,65.0,06:29,389


In [249]:
# UniDiasABP
# UniDiasABP_Initial
# UniDiasABP_Min
# UniDiasABP_Max
# UniDiasABP_Mean
# UniDiasABP_SD x
# UniDiasABP_Time_Duration - threshold < 40 x
# UniDiasABP_Count
# UniDiasABP_Slope
# UniDiasABP_Miss
# UniDiasABP_Last_Minute


def unidiasabp_stats(df_sub, df_record):
  col = 'UniDiasABP'
  col_initial = col+'_Initial'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  #col_sd = col+'_SD'
  #col_td = col+'_Time_Duration'
  #threshold = 40
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    #df_record[col_sd] = np.nan
    #df_record[col_td] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      #df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      #df_record[col_sd] = 0
      #df_record[col_td] = 0
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      #df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      #df_record[col_sd] = df_sub['Value'].std()
      #max_duration, max_begin, max_end = find_max_duration(df_sub, threshold, 'below')
      #time_duration, time_begin, time_end = find_time_duration(df_sub, max_begin, max_end)
      #df_record[col_td] = time_duration
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [250]:
unidiasabp_stats(df_sub, df_record)

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute,SysABP_Initial,SysABP_Min,SysABP_Max,SysABP_Mean,SysABP_SD,SysABP_Count,SysABP_Slope,SysABP_Miss,SysABP_Last_Minute,TroponinI_Max,TroponinI_Count,TroponinI_Slope,TroponinI_Miss,TroponinI_Last_Minute,TroponinT_Max,TroponinT_Count,TroponinT_Slope,TroponinT_Miss,TroponinT_Last_Minute
0,141669,2.3,2.2,2.

In [251]:
df1 = df_temp[df_temp['Parameter'] == 'MAP'].reset_index(drop=True)
df1

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,MAP,81.0,17:59,1079
1,137188,MAP,85.0,18:29,1109
2,137188,MAP,87.0,18:59,1139
3,137188,MAP,80.0,19:29,1169
4,137188,MAP,76.0,19:44,1184
5,137188,MAP,64.0,19:54,1194
6,137188,MAP,65.0,19:59,1199
7,137188,MAP,75.0,20:29,1229
8,137188,MAP,75.0,21:29,1289
9,137188,MAP,84.0,22:29,1349


In [252]:
df2 = df_temp[df_temp['Parameter'] == 'NIMAP'].reset_index(drop=True)
df2

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,NIMAP,69.0,02:14,134
1,137188,NIMAP,70.0,02:29,149
2,137188,NIMAP,75.0,02:44,164
3,137188,NIMAP,80.0,02:59,179
4,137188,NIMAP,81.0,03:14,194
5,137188,NIMAP,83.0,03:29,209
6,137188,NIMAP,76.0,03:59,239
7,137188,NIMAP,76.0,04:29,269
8,137188,NIMAP,76.0,05:29,329
9,137188,NIMAP,75.0,06:29,389


In [253]:
df_sub = find_unidf(df1, df2)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,UniMAP,69.0,02:14,134
1,137188,UniMAP,70.0,02:29,149
2,137188,UniMAP,75.0,02:44,164
3,137188,UniMAP,80.0,02:59,179
4,137188,UniMAP,81.0,03:14,194
5,137188,UniMAP,83.0,03:29,209
6,137188,UniMAP,76.0,03:59,239
7,137188,UniMAP,76.0,04:29,269
8,137188,UniMAP,76.0,05:29,329
9,137188,UniMAP,75.0,06:29,389


In [254]:
# UniMAP
# UniMAP_Initial
# UniMAP_Final
# UniMAP_Min
# UniMAP_Max
# UniMAP_Mean
# UniMAP_SD
# UniMAP_Time_Duration - threshold < 65 x
# UniMAP_Count
# UniMAP_Slope
# UniMAP_Miss
# UniMAP_Last_Minute


def unimap_stats(df_sub, df_record):
  col = 'UniMAP'
  col_initial = col+'_Initial'
  col_final = col+'_Final'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_sd = col+'_SD'
  #col_td = col+'_Time_Duration'
  #threshold = 65
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_final] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_sd] = np.nan
    #df_record[col_td] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = 0
      #df_record[col_td] = 0
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = df_sub['Value'].std()
      #max_duration, max_begin, max_end = find_max_duration(df_sub, threshold, 'below')
      #time_duration, time_begin, time_end = find_time_duration(df_sub, max_begin, max_end)
      #df_record[col_td] = time_duration
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [255]:
unimap_stats(df_sub, df_record)

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute,SysABP_Initial,SysABP_Min,SysABP_Max,SysABP_Mean,SysABP_SD,SysABP_Count,SysABP_Slope,SysABP_Miss,SysABP_Last_Minute,TroponinI_Max,TroponinI_Count,TroponinI_Slope,TroponinI_Miss,TroponinI_Last_Minute,TroponinT_Max,TroponinT_Count,TroponinT_Slope,TroponinT_Miss,TroponinT_Last_Minute
0,141669,2.3,2.2,2.

In [256]:
df1 = df_temp[df_temp['Parameter'] == 'SysABP'].reset_index(drop=True)
df1

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,SysABP,112.0,17:59,1079
1,137188,SysABP,112.0,18:29,1109
2,137188,SysABP,124.0,18:59,1139
3,137188,SysABP,115.0,19:29,1169
4,137188,SysABP,107.0,19:44,1184
5,137188,SysABP,92.0,19:54,1194
6,137188,SysABP,96.0,19:59,1199
7,137188,SysABP,102.0,20:29,1229
8,137188,SysABP,103.0,21:29,1289
9,137188,SysABP,121.0,22:29,1349


In [257]:
df2 = df_temp[df_temp['Parameter'] == 'NISysABP'].reset_index(drop=True)
df2

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,NISysABP,97.0,02:14,134
1,137188,NISysABP,98.0,02:29,149
2,137188,NISysABP,107.0,02:44,164
3,137188,NISysABP,112.0,02:59,179
4,137188,NISysABP,111.0,03:14,194
5,137188,NISysABP,111.0,03:29,209
6,137188,NISysABP,106.0,03:59,239
7,137188,NISysABP,108.0,04:29,269
8,137188,NISysABP,111.0,05:29,329
9,137188,NISysABP,114.0,06:29,389


In [258]:
df_sub = find_unidf(df1, df2)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,UniSysABP,97.0,02:14,134
1,137188,UniSysABP,98.0,02:29,149
2,137188,UniSysABP,107.0,02:44,164
3,137188,UniSysABP,112.0,02:59,179
4,137188,UniSysABP,111.0,03:14,194
5,137188,UniSysABP,111.0,03:29,209
6,137188,UniSysABP,106.0,03:59,239
7,137188,UniSysABP,108.0,04:29,269
8,137188,UniSysABP,111.0,05:29,329
9,137188,UniSysABP,114.0,06:29,389


In [259]:
# UniSysABP
# UniSysABP_Initial
# UniSysABP_Final
# UniSysABP_Min
# UniSysABP_Max
# UniSysABP_Mean
# UniSysABP_SD
# UniSysABP_Time_Duration - threshold < 90 x
# UniSysABP_Count
# UniSysABP_Slope
# UniSysABP_Miss
# UniSysABP_Last_Minute


def unisysabp_stats(df_sub, df_record):
  col = 'UniSysABP'
  col_initial = col+'_Initial'
  col_final = col+'_Final'
  col_min = col+'_Min'
  col_max = col+'_Max'
  col_mean = col+'_Mean'
  col_sd = col+'_SD'
  #col_td = col+'_Time_Duration'
  #threshold = 90
  col_count = col+'_Count'
  col_slope = col+'_Slope'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_initial] = np.nan
    df_record[col_final] = np.nan
    df_record[col_min] = np.nan
    df_record[col_max] = np.nan
    df_record[col_mean] = np.nan
    df_record[col_sd] = np.nan
    #df_record[col_td] = np.nan
    df_record[col_count] = 0
    df_record[col_slope] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = 0
      #df_record[col_td] = 0
      df_record[col_count] = 1
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_initial] = np.nan
      df_record[col_final] = np.nan
      df_record[col_min] = np.nan
      df_record[col_max] = np.nan
      df_record[col_mean] = np.nan
      df_record[col_sd] = np.nan
      #df_record[col_td] = np.nan
      df_record[col_count] = 0
      df_record[col_slope] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_initial] = df_sub['Value'].iloc[0]
      df_record[col_final] = df_sub['Value'].iloc[-1]
      df_record[col_min] = df_sub['Value'].min()
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_mean] = df_sub['Value'].mean()
      df_record[col_sd] = df_sub['Value'].std()
      #max_duration, max_begin, max_end = find_max_duration(df_sub, threshold, 'below')
      #time_duration, time_begin, time_end = find_time_duration(df_sub, max_begin, max_end)
      #df_record[col_td] = time_duration
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_slope] = compute_regression_slope(df_sub)
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [260]:
unisysabp_stats(df_sub, df_record)

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute,SysABP_Initial,SysABP_Min,SysABP_Max,SysABP_Mean,SysABP_SD,SysABP_Count,SysABP_Slope,SysABP_Miss,SysABP_Last_Minute,TroponinI_Max,TroponinI_Count,TroponinI_Slope,TroponinI_Miss,TroponinI_Last_Minute,TroponinT_Max,TroponinT_Count,TroponinT_Slope,TroponinT_Miss,TroponinT_Last_Minute
0,141669,2.3,2.2,2.

In [261]:
df1 = df_temp[df_temp['Parameter'] == 'TroponinI'].reset_index(drop=True)
df1

,RecordID,Parameter,Value,Time,Time_Minute


In [262]:
df2 = df_temp[df_temp['Parameter'] == 'TroponinT'].reset_index(drop=True)
df2

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,TroponinT,0.40,03:22,202
1,137188,TroponinT,0.32,11:50,710
2,137188,TroponinT,0.22,22:00,1320


In [263]:
df_sub = find_troponin(df1, df2)
df_sub

,RecordID,Parameter,Value,Time,Time_Minute
0,137188,Troponin,0.40,03:22,202
1,137188,Troponin,0.32,11:50,710
2,137188,Troponin,0.22,22:00,1320


In [264]:
# Troponin
# Troponin_Max
# Troponin_Count
# Troponin_Delta
# Troponin_Miss
# Troponin_Last_Minute


def troponin_stats(df_sub, df_record):
  col = 'Troponin'
  col_max = col+'_Max'
  col_count = col+'_Count'
  col_delta = col+'_Delta'
  col_miss = col+'_Miss'
  col_last_minute = col+'_Last_Minute'

  if len(df_sub) == 0:
    df_record[col_max] = np.nan
    df_record[col_count] = 0
    df_record[col_delta] = 0
    df_record[col_miss] = 1
    df_record[col_last_minute] = 0
  elif len(df_sub) == 1:
    if df_sub.isnull().sum().sum() != 0:
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]
  else:
    if df_sub.isnull().sum().sum() == len(df_sub):
      df_record[col_max] = np.nan
      df_record[col_count] = 0
      df_record[col_delta] = 0
      df_record[col_miss] = 1
      df_record[col_last_minute] = 0
    else:
      df_record[col_max] = df_sub['Value'].max()
      df_record[col_count] = df_sub['Value'].count()
      df_record[col_delta] = df_sub['Value'].iloc[-1] - df_sub['Value'].iloc[0]
      df_record[col_miss] = 0
      df_record[col_last_minute] = df_sub['Time_Minute'].iloc[-1]

  return df_record

In [265]:
troponin_stats(df_sub, df_record)

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,DiasABP_Initial,DiasABP_Min,DiasABP_Max,DiasABP_Mean,DiasABP_SD,DiasABP_Count,DiasABP_Slope,DiasABP_Miss,DiasABP_Last_Minute,MAP_Initial,MAP_Min,MAP_Max,MAP_Mean,MAP_SD,MAP_Count,MAP_Slope,MAP_Miss,MAP_Last_Minute,NIDiasABP_Initial,NIDiasABP_Min,NIDiasABP_Max,NIDiasABP_Mean,NIDiasABP_SD,NIDiasABP_Count,NIDiasABP_Slope,NIDiasABP_Miss,NIDiasABP_Last_Minute,NIMAP_Initial,NIMAP_Min,NIMAP_Max,NIMAP_Mean,NIMAP_SD,NIMAP_Count,NIMAP_Slope,NIMAP_Miss,NIMAP_Last_Minute,NISysABP_Initial,NISysABP_Min,NISysABP_Max,NISysABP_Mean,NISysABP_SD,NISysABP_Count,NISysABP_Slope,NISysABP_Miss,NISysABP_Last_Minute,SysABP_Initial,SysABP_Min,SysABP_Max,SysABP_Mean,SysABP_SD,SysABP_Count,SysABP_Slope,SysABP_Miss,SysABP_Last_Minute,TroponinI_Max,TroponinI_Count,TroponinI_Slope,TroponinI_Miss,TroponinI_Last_Minute,TroponinT_Max,TroponinT_Count,TroponinT_Slope,TroponinT_Miss,TroponinT_Last_Minute
0,141669,2.3,2.2,2.

# extract statistics features for the whole dataset

In [266]:
df_record_all = pd.DataFrame(columns=col_df)
df_record_all.loc[0] = np.nan

for i in range(len(dfs_nan)):
  df_temp = dfs_nan[i]
  df_record = pd.DataFrame(columns=col_df)
  df_record.loc[0] = np.nan
  df_record['RecordID'] = df_temp['RecordID'].iloc[0]

  for col in col_icu:
    if col == 'Albumin':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      albumin_stats(df_sub, df_record, col)
    elif col == 'ALP':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      alp_stats(df_sub, df_record, col)
    elif col == 'ALT':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      alt_stats(df_sub, df_record, col)
    elif col == 'AST':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      ast_stats(df_sub, df_record, col)
    elif col == 'Bilirubin':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      bilirubin_stats(df_sub, df_record, col)
    elif col == 'BUN':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      df_sub2 = df_temp[df_temp['Parameter'] == 'Creatinine'].reset_index(drop=True)
      bun_stats(df_sub, df_sub2, df_record, col)
    elif col == 'Cholesterol':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      cholesterol_stats(df_sub, df_record, col)
    elif col == 'Creatinine':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      creatinine_stats(df_sub, df_record, col)
    elif col == 'DiasABP':
      col2 = 'NI'+col
      df1 = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      df2 = df_temp[df_temp['Parameter'] == col2].reset_index(drop=True)
      df_sub = find_unidf(df1, df2)
      #df_record['DiasABP_MAD'] = find_mad(df1, df2)
      unidiasabp_stats(df_sub, df_record)
    elif col == 'FiO2':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      fio2_stats(df_sub, df_record, col)
    elif col == 'GCS':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      gcs_stats(df_sub, df_record, col)
    elif col == 'Glucose':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      glucose_stats(df_sub, df_record, col)
    elif col == 'HCO3':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      hco3_stats(df_sub, df_record, col)
    elif col == 'HCT':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      hct_stats(df_sub, df_record, col)
    elif col == 'HR':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      hr_stats(df_sub, df_record, col)
    elif col == 'K':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      k_stats(df_sub, df_record, col)
    elif col == 'Lactate':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      lactate_stats(df_sub, df_record, col)
    elif col == 'Mg':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      mg_stats(df_sub, df_record, col)
    elif col == 'MAP':
      col2 = 'NI'+col
      df1 = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      df2 = df_temp[df_temp['Parameter'] == col2].reset_index(drop=True)
      df_sub = find_unidf(df1, df2)
      #df_record['MAP_MAD'] = find_mad(df1, df2)
      unimap_stats(df_sub, df_record)
    elif col == 'MechVent':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      mechvent_stats(df_sub,df_record, col)
    elif col == 'Na':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      na_stats(df_sub,df_record, col)
    elif col == 'NIDiasABP':
      pass
    elif col == 'NIMAP':
      pass
    elif col == 'NISysABP':
      pass
    elif col == 'PaCO2':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      paco2_stats(df_sub, df_record, col)
    elif col == 'PaO2':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      df_sub2 = df_temp[df_temp['Parameter'] == 'FiO2'].reset_index(drop=True)
      pao2_stats(df_sub, df_sub2, df_record, col)
    elif col == 'pH':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      ph_stats(df_sub, df_record, col)
    elif col == 'Platelets':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      platelets_stats(df_sub, df_record, col)
    elif col == 'RespRate':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      resprate_stats(df_sub, df_record, col)
    elif col == 'SaO2':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      sao2_stats(df_sub, df_record, col)
    elif col == 'SysABP':
      col2 = 'NI'+col
      df1 = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      df2 = df_temp[df_temp['Parameter'] == col2].reset_index(drop=True)
      df_sub = find_unidf(df1, df2)
      #df_record['SysABP_MAD'] = find_mad(df1, df2)
      unisysabp_stats(df_sub, df_record)
    elif col == 'Temp':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      temp_stats(df_sub, df_record, col)
    elif col == 'TroponinI':
      df1 = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      df2 = df_temp[df_temp['Parameter'] == 'TroponinT'].reset_index(drop=True)
      df_sub = find_troponin(df1, df2)
      troponin_stats(df_sub, df_record)
    elif col == 'TroponinT':
      pass
    elif col == 'Urine':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      df_sub2 = df_temp[df_temp['Parameter'] == 'Weight'].reset_index(drop=True)
      urine_stats(df_sub, df_sub2, df_record, col)
    elif col == 'WBC':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      wbc_stats(df_sub, df_record, col)
    elif col == 'Weight':
      df_sub = df_temp[df_temp['Parameter'] == col].reset_index(drop=True)
      weight_stats(df_sub, df_record, col)

  df_record_all = pd.concat([df_record_all, df_record], ignore_index=True)

# final dataframe

In [267]:
df_record_all = df_record_all.drop(index=0)
df_record_all.shape

(12000, 243)

In [268]:
df_record_all['RecordID'] = df_record_all['RecordID'].astype(int)

In [269]:
df_record_all.dtypes

RecordID                        int64
Albumin_Initial               float64
Albumin_Min                   float64
Albumin_Max                   float64
Albumin_Mean                  float64
ALP_Initial                   float64
ALP_Max                       float64
ALT_Initial                   float64
ALT_Max                       float64
AST_Initial                   float64
AST_Max                       float64
Bilirubin_Initial             float64
Bilirubin_Max                 float64
BUN_Initial                   float64
BUN_Max                       float64
BUN_Mean                      float64
BUN_Creatinine_Ratio          float64
Cholesterol_Initial           float64
Cholesterol_Min               float64
Creatinine_Initial            float64
Creatinine_Max                float64
Creatinine_Mean               float64
FiO2_Initial                  float64
FiO2_Max                      float64
FiO2_Mean                     float64
GCS_Initial                   float64
GCS_Final   

In [270]:
df_combined = pd.merge(left=df_record_all, right=df_patient, on='RecordID', how='inner')
df_combined.shape

(12000, 255)

In [271]:
(df_combined.isnull().sum()/len(df_combined)*100).sort_values(ascending=False)

Cholesterol_Min               92.100000
Cholesterol_Initial           92.100000
Troponin_Max                  73.325000
RespRate_Initial              72.316667
RespRate_Mean                 72.266667
RespRate_Max                  72.266667
Lactate_Clearance_Rate_12H    69.041667
Lactate_Clearance_Rate_6H     65.150000
Albumin_Max                   59.408333
Albumin_Mean                  59.408333
Albumin_Initial               59.408333
Albumin_Min                   59.408333
ALP_Initial                   57.525000
ALP_Max                       57.525000
AST_Initial                   56.566667
ALT_Max                       56.558333
Bilirubin_Initial             56.558333
Bilirubin_Max                 56.558333
ALT_Initial                   56.558333
AST_Max                       56.550000
SaO2_Initial                  55.350000
SaO2_Min                      55.316667
Height                        47.825000
Lactate_Initial               45.200000
Lactate_Max                   45.200000


In [272]:
df_combined.head(10)

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,Age,Gender,Height,SAPS-I,SOFA,Length_of_stay,Survival,In-hospital_death,ICUType_1.0,ICUType_2.0,ICUType_3.0,ICUType_4.0
0,137188,2.3,2.2,2.3,2.266667,84.0,84.0,20.0,20.0,123.0,123.0,6.6,11.2,7.0,7.0,5.500000,10.000000,NaN,NaN,0.9,0.9,0.675000,1.0,1.0,0.655556,3.0,7.0,3.0,7.0,5.615385,101.0,88.0,173.0,127.500000,22.0,25.0,22.0,25.0,29.7,26.6,29.7,28.416667,109.0,106.0,88.0,139.0,109.265625,12.787943,4.0,4.6,3.6,4.6,4.060000,1.5,1.8,0.000000,-20.000000,1.0,2.2,1.7,2.2,136.0,136.0,136.0,139.0,136.750000,40.0,39.0,51.0,187.0,73.0,121.666667,7.39,7.29,7.42,7.361429,85.0,71.0,NaN,NaN,NaN,NaN,NaN,37.6,36.3,38.9,37.329412,0.645041,2763.0,0.167785,3.775168,0.934510,13.3,9.3,13.3,59.60,59.6,60.0,38.0,77.0,59.859375,69.00,86.00,48.00,95.00,75.156250,8.942885,97.0,123.0,82.0,131.0,108.609375,10.529802,0.4,3.0,3.0,3.0,3.0,4.0,3.0,0.0,4.0,18.0,13.0,4.0,4.0,6.0,64.0,5.0,6.0,18.0,4.0,4.0,7.0,7.0,7.0,4.

In [273]:
df_combined.describe()

,RecordID,Albumin_Initial,Albumin_Min,Albumin_Max,Albumin_Mean,ALP_Initial,ALP_Max,ALT_Initial,ALT_Max,AST_Initial,AST_Max,Bilirubin_Initial,Bilirubin_Max,BUN_Initial,BUN_Max,BUN_Mean,BUN_Creatinine_Ratio,Cholesterol_Initial,Cholesterol_Min,Creatinine_Initial,Creatinine_Max,Creatinine_Mean,FiO2_Initial,FiO2_Max,FiO2_Mean,GCS_Initial,GCS_Final,GCS_Min,GCS_Max,GCS_Mean,Glucose_Initial,Glucose_Min,Glucose_Max,Glucose_Mean,HCO3_Initial,HCO3_Final,HCO3_Min,HCO3_Max,HCT_Initial,HCT_Min,HCT_Max,HCT_Mean,HR_Initial,HR_Final,HR_Min,HR_Max,HR_Mean,HR_SD,K_Initial,K_Final,K_Min,K_Max,K_Mean,Lactate_Initial,Lactate_Max,Lactate_Clearance_Rate_6H,Lactate_Clearance_Rate_12H,MechVent_Any,Mg_Initial,Mg_Min,Mg_Max,Na_Initial,Na_Final,Na_Min,Na_Max,Na_Mean,PaCO2_Initial,PaCO2_Min,PaCO2_Max,PaO2_Initial,PaO2_Min,PaO2_FiO2_Ratio,pH_Initial,pH_Min,pH_Max,pH_Mean,Platelets_Initial,Platelets_Min,RespRate_Initial,RespRate_Max,RespRate_Mean,SaO2_Initial,SaO2_Min,Temp_Initial,Temp_Min,Temp_Max,Temp_Mean,Temp_SD,Urine_Sum,Urine_MinRate,Urine_MaxRate,Urine_MeanRate,WBC_Initial,WBC_Min,WBC_Max,Weight_Initial,Weight_Final,UniDiasABP_Initial,UniDiasABP_Min,UniDiasABP_Max,UniDiasABP_Mean,UniMAP_Initial,UniMAP_Final,UniMAP_Min,UniMAP_Max,UniMAP_Mean,UniMAP_SD,UniSysABP_Initial,UniSysABP_Final,UniSysABP_Min,UniSysABP_Max,UniSysABP_Mean,UniSysABP_SD,Troponin_Max,Albumin_Count,ALP_Count,ALT_Count,AST_Count,BUN_Count,Bilirubin_Count,Cholesterol_Count,Creatinine_Count,FiO2_Count,GCS_Count,Glucose_Count,HCO3_Count,HCT_Count,HR_Count,K_Count,Lactate_Count,MechVent_Count,Mg_Count,Na_Count,PaCO2_Count,PaO2_Count,pH_Count,Platelets_Count,RespRate_Count,SaO2_Count,Temp_Count,Urine_Count,WBC_Count,UniDiasABP_Count,UniMAP_Count,UniSysABP_Count,Troponin_Count,Albumin_Delta,ALP_Delta,ALT_Delta,AST_Delta,Bilirubin_Delta,BUN_Delta,Cholesterol_Delta,Creatinine_Delta,HCO3_Delta,HCT_Delta,K_Delta,Mg_Delta,Na_Delta,PaO2_Delta,Platelets_Delta,SaO2_Delta,WBC_Delta,Weight_Delta,Troponin_Delta,FiO2_Slope,GCS_Slope,Glucose_Slope,HR_Slope,PaCO2_Slope,pH_Slope,RespRate_Slope,Temp_Slope,Urine_Rate_Slope,UniDiasABP_Slope,UniMAP_Slope,UniSysABP_Slope,Albumin_Miss,ALP_Miss,ALT_Miss,AST_Miss,Bilirubin_Miss,BUN_Miss,Cholesterol_Miss,Creatinine_Miss,FiO2_Miss,GCS_Miss,Glucose_Miss,HCO3_Miss,HCT_Miss,HR_Miss,K_Miss,Lactate_Miss,MechVent_Miss,Mg_Miss,Na_Miss,PaCO2_Miss,PaO2_Miss,pH_Miss,Platelets_Miss,RespRate_Miss,SaO2_Miss,Temp_Miss,Urine_Miss,WBC_Miss,Weight_Miss,UniDiasABP_Miss,UniMAP_Miss,UniSysABP_Miss,Troponin_Miss,Albumin_Last_Minute,ALP_Last_Minute,ALT_Last_Minute,AST_Last_Minute,Bilirubin_Last_Minute,BUN_Last_Minute,Cholesterol_Last_Minute,Creatinine_Last_Minute,FiO2_Last_Minute,GCS_Last_Minute,Glucose_Last_Minute,HCO3_Last_Minute,HCT_Last_Minute,HR_Last_Minute,K_Last_Minute,Lactate_Last_Minute,MechVent_Last_Minute,Mg_Last_Minute,Na_Last_Minute,PaCO2_Last_Minute,PaO2_Last_Minute,pH_Last_Minute,Platelets_Last_Minute,RespRate_Last_Minute,SaO2_Last_Minute,Temp_Last_Minute,Urine_Last_Minute,WBC_Last_Minute,Weight_Last_Minute,UniDiasABP_Last_Minute,UniMAP_Last_Minute,UniSysABP_Last_Minute,Troponin_Last_Minute,Age,Gender,Height,SAPS-I,SOFA,Length_of_stay,Survival,In-hospital_death
count,12000.000000,4871.000000,4871.000000,4871.000000,4871.000000,5097.000000,5097.000000,5213.000000,5213.000000,5212.000000,5214.000000,5213.000000,5213.000000,11816.000000,11817.000000,11817.000000,11817.000000,948.000000,948.000000,11817.000000,11817.000000,11817.000000,8116.000000,8116.000000,8116.000000,11815.000000,11815.000000,11815.000000,11815.000000,11815.000000,11703.00000,11706.000000,11706.000000,11706.000000,11793.000000,11793.000000,11793.000000,11793.000000,11808.000000,11810.000000,11810.000000,11810.000000,11816.000000,11802.000000,11816.000000,11816.000000,11816.000000,11816.000000,11750.000000,11751.000000,11751.000000,11751.000000,11751.000000,6576.000000,6576.000000,4182.000000,3715.000000,7581.0,11699.000000,11708.000000,11708.000000,11784.000000,11788.000000,11788.000000,11788.000000,11788.

In [274]:
df_combined.columns.tolist()

['RecordID',
 'Albumin_Initial',
 'Albumin_Min',
 'Albumin_Max',
 'Albumin_Mean',
 'ALP_Initial',
 'ALP_Max',
 'ALT_Initial',
 'ALT_Max',
 'AST_Initial',
 'AST_Max',
 'Bilirubin_Initial',
 'Bilirubin_Max',
 'BUN_Initial',
 'BUN_Max',
 'BUN_Mean',
 'BUN_Creatinine_Ratio',
 'Cholesterol_Initial',
 'Cholesterol_Min',
 'Creatinine_Initial',
 'Creatinine_Max',
 'Creatinine_Mean',
 'FiO2_Initial',
 'FiO2_Max',
 'FiO2_Mean',
 'GCS_Initial',
 'GCS_Final',
 'GCS_Min',
 'GCS_Max',
 'GCS_Mean',
 'Glucose_Initial',
 'Glucose_Min',
 'Glucose_Max',
 'Glucose_Mean',
 'HCO3_Initial',
 'HCO3_Final',
 'HCO3_Min',
 'HCO3_Max',
 'HCT_Initial',
 'HCT_Min',
 'HCT_Max',
 'HCT_Mean',
 'HR_Initial',
 'HR_Final',
 'HR_Min',
 'HR_Max',
 'HR_Mean',
 'HR_SD',
 'K_Initial',
 'K_Final',
 'K_Min',
 'K_Max',
 'K_Mean',
 'Lactate_Initial',
 'Lactate_Max',
 'Lactate_Clearance_Rate_6H',
 'Lactate_Clearance_Rate_12H',
 'MechVent_Any',
 'Mg_Initial',
 'Mg_Min',
 'Mg_Max',
 'Na_Initial',
 'Na_Final',
 'Na_Min',
 'Na_Max',
 

In [275]:
df_combined.to_csv(path+'/df_clean.csv', index=False)